In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os
import subprocess
import time
import pandas as pd
import pickle
import math
import scipy.stats as stats

import scipy.io as sio
from scipy import interpolate
from sklearn.metrics import roc_auc_score as auROC
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.colorbar as colorbar
import matplotlib.patches as mpatches
import seaborn as sns
import sys
import re
import random
from scipy.stats import ttest_ind
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import numpy as np
from pathlib import Path  # Added this import

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.stats as stats

# --- Helper Functions ---

# Load MATLAB files
def find_matfiles(indir):
    tempfiles = next(os.walk(indir))[2]
    matfiles = [f for f in tempfiles if os.path.splitext(f)[1] == '.mat']
    return sorted(matfiles)

# Extract cues from the event log
def extract_cues(eventlog):
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    totalcues = np.concatenate((sucrosecues, nosucrosecues))
    return sucrosecues, nosucrosecues, totalcues

# Calculate delta lick rate for a given set of cues and offsets, baseline is 3s, cue period is 3s, reward is 3s
def calculate_delta_lickrate(cues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000)):
    baseline_start = cues + baseline_offset[0]
    baseline_end = cues + baseline_offset[1]
    period_start = cues + period_offset[0]
    period_end = cues + period_offset[1]

    baseline_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(baseline_start, baseline_end)])
    period_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(period_start, period_end)])

    baseline_hz = baseline_licking / abs(baseline_offset[1] - baseline_offset[0]) * 1000
    period_hz = period_licking / (period_offset[1] - period_offset[0]) * 1000

    delta_lickrate = period_hz - baseline_hz
    return delta_lickrate, np.mean(delta_lickrate)

# Find session length in minutes
def find_session_length(eventlog):
    sessionlength = max(eventlog[:, 1])
    return round((sessionlength / 1000 / 60), 2)

# Calculate normalized auROC and p-value using Mann-Whitney U
def calculate_auROC(x, y, phase='Acquisition', offset_to_zero=True):
    # Handle edge cases where a behavioral file might be missing trials
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan
    
    # Calculate Mann-Whitney U and p-value
    U, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    
    # Calculate raw auROC directly from the U statistic
    A = U / (len(x) * len(y))

    # Apply Reversal Rule Mastery logic
    if phase == 'Reversal':
        A = 1 - A

    # Scale to -1 to 1 (0 = chance) if requested
    if offset_to_zero:
        return (2 * (A - 0.5), p)
    else:
        return (A, p)

# --- Main Script ---

# Directories and parameters
basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes\Reversal'
datafolder = 'REV-D20'

# Auto-detect Phase based on folder name
if 'REV-' in datafolder:
    current_phase = 'Reversal'
elif 'PC-' in datafolder:
    current_phase = 'Acquisition'
else:
    print(f"[!] Warning: Could not strictly determine phase from '{datafolder}'. Defaulting to Acquisition.")
    current_phase = 'Acquisition'

print(f"Running Analysis for Folder: {datafolder}")
print(f"Auto-Detected Phase: {current_phase}")

# Output file
output_file = os.path.join(basedir, 'Output CSVs', f'{datafolder}_dLickRate-avg_auROC.csv')

# DataFrame column names (Added normalized auROC and p-value columns)
column_names = ['ID', 'Session Length', 'Total Cues', 'Total CS+', 'Total CS-', 'Sucrose Rewards',
                'CS+ avg dLickrate Cue', 'CS- avg dLickrate Cue',
                'CS+ avg dLickrate Reward', 'CS- avg dLickrate Reward',
                'auROC_CSplus_vs_CSminus_Norm', 'auROC_pval']

# Load MATLAB files
matfiles = find_matfiles(os.path.join(basedir, datafolder))
print(f'There are {len(matfiles)} files')

# Initialize list to collect rows
rows = []

for matfile in matfiles:
    # Load behavioral data
    behaviordata = sio.loadmat(os.path.join(basedir, datafolder, matfile))
    eventlog = np.squeeze(behaviordata['eventlog'])
    licks = np.squeeze(behaviordata['licks'])
    fxdpumps = np.squeeze(behaviordata['fxdpumps'])

    # Extract cues
    sucrosecues, nosucrosecues, totalcues = extract_cues(eventlog)

    # CS+ cue period
    sucrosecues_delta_lickrate, sucrosecues_delta_lickrate_avg = calculate_delta_lickrate(
        sucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000))

    # CS+ reward period
    sucrosecues_rewarded_delta_lickrate, sucrosecues_rewarded_delta_lickrate_avg = calculate_delta_lickrate(
        sucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(3000, 6000))

    # CS- cue period
    nosucrosecues_delta_lickrate, nosucrosecues_delta_lickrate_avg = calculate_delta_lickrate(
        nosucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000))

    # CS- reward period (omission)
    nosucrosecues_omitted_delta_lickrate, nosucrosecues_omitted_delta_lickrate_avg = calculate_delta_lickrate(
        nosucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(3000, 6000))

    # Session length
    session_length_in_minutes = find_session_length(eventlog)

    # --- Calculate Normalized auROC & p-value for Discrimination ---
    # Passing in the trial-by-trial arrays and the auto-detected phase
    auroc_discrimination_norm, auroc_pval = calculate_auROC(
        sucrosecues_delta_lickrate, 
        nosucrosecues_delta_lickrate, 
        phase=current_phase,
        offset_to_zero=True
    )

    # Append row to list
    rows.append({
        'ID': matfile,
        'Session Length': session_length_in_minutes,
        'Total Cues': len(totalcues),
        'Total CS+': len(sucrosecues),
        'Total CS-': len(nosucrosecues),
        'Sucrose Rewards': len(fxdpumps),
        'CS+ avg dLickrate Cue': sucrosecues_delta_lickrate_avg,
        'CS- avg dLickrate Cue': nosucrosecues_delta_lickrate_avg,
        'CS+ avg dLickrate Reward': sucrosecues_rewarded_delta_lickrate_avg,
        'CS- avg dLickrate Reward': nosucrosecues_omitted_delta_lickrate_avg,
        'auROC_CSplus_vs_CSminus_Norm': auroc_discrimination_norm,
        'auROC_pval': auroc_pval
    })

# Build DataFrame once after the loop
df = pd.DataFrame(rows, columns=column_names)
df.set_index('ID', inplace=True)

# Export DataFrame to CSV
os.makedirs(os.path.dirname(output_file), exist_ok=True) # Ensures the output directory exists
df.to_csv(output_file)

# Print summary
print("\n--- Preview of Processed Data ---")
print(df.head())
print(f'\nData saved to {output_file}')

In [ ]:
#####updating the code to include the dictionary structure used for paper analysis

In [ ]:
import os
import numpy as np
import scipy.io as sio
import scipy.stats as stats
from pathlib import Path

# --- Helper Functions ---

def extract_cues(eventlog):
    """Extract CS+ and CS- timestamps from the event log."""
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    totalcues = np.concatenate((sucrosecues, nosucrosecues))
    return sucrosecues, nosucrosecues, totalcues

def calculate_delta_lickrate(cues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000)):
    """Calculate delta lick rate for a given set of cues and offsets."""
    if len(cues) == 0:
        return np.array([]), np.nan
        
    baseline_start = cues + baseline_offset[0]
    baseline_end = cues + baseline_offset[1]
    period_start = cues + period_offset[0]
    period_end = cues + period_offset[1]

    baseline_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(baseline_start, baseline_end)])
    period_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(period_start, period_end)])

    baseline_hz = baseline_licking / abs(baseline_offset[1] - baseline_offset[0]) * 1000
    period_hz = period_licking / (period_offset[1] - period_offset[0]) * 1000

    delta_lickrate = period_hz - baseline_hz
    return delta_lickrate, np.mean(delta_lickrate)

def find_session_length(eventlog):
    """Find session length in minutes."""
    sessionlength = max(eventlog[:, 1])
    return round((sessionlength / 1000 / 60), 2)

def calculate_auROC(x, y, phase, offset_to_zero=True):
    """
    Calculate normalized auROC and p-value using Mann-Whitney U.
    Automatically applies (1 - A) logic if the phase is 'Reversal'.
    """
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan
    
    U, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    A = U / (len(x) * len(y))

    # Apply the Reversal mathematical flip right here
    if phase == 'Reversal':
        A = 1 - A

    # Normalize to -1 to 1
    if offset_to_zero:
        return (2 * (A - 0.5), p)
    else:
        return (A, p)

# --- Main Dictionary Builder ---

def build_cbx_data_dictionary(basedir):
    """
    Build hierarchical data structure from CBX MATLAB files.
    Traverses: basedir / Phase (Acquisition or Reversal) / Session (e.g., PC-D1) / .mat files
    """
    # Initialize the top-level structure
    data_dict = {
        'Acquisition': {'CBX': {}, 'SAL': {}},
        'Reversal': {'CBX': {}, 'SAL': {}}
    }
    
    basedir_path = Path(basedir)
    
    for phase in ['Acquisition', 'Reversal']:
        print(f"\nProcessing {phase} phase...")
        phase_dir = basedir_path / phase
        
        # Skip if the phase directory doesn't exist yet
        if not phase_dir.exists():
            print(f"  Directory not found: {phase_dir}. Skipping.")
            continue
            
        # Iterate through session folders (e.g., PC-D1, PC-D2)
        for session_dir in phase_dir.iterdir():
            if not session_dir.is_dir() or session_dir.name == 'Output CSVs':
                continue
                
            session_id = session_dir.name
            print(f"  Processing session: {session_id}")
            
            # Initialize the session dictionary for both conditions
            if session_id not in data_dict[phase]['CBX']:
                data_dict[phase]['CBX'][session_id] = {}
            if session_id not in data_dict[phase]['SAL']:
                data_dict[phase]['SAL'][session_id] = {}
                
            # Find all .mat files in the session folder
            for matfile in session_dir.glob('*.mat'):
                filename = matfile.name
                
                # --- NEW Bulletproof Filename Parsing ---
                # Example: PFC-CBX10_PC-D1_CBX_0_6000_260311-190731.mat
                clean_name = filename.replace('.mat', '')
                parts = clean_name.split('_')
                
                try:
                    # Find where the session ID is in the filename chunks
                    session_idx = parts.index(session_id)
                    
                    # Everything before the session ID is the animal name
                    animal_id = '_'.join(parts[:session_idx])
                    
                    # The chunk immediately following the session ID is the condition
                    condition = parts[session_idx + 1].upper()
                    
                except (ValueError, IndexError):
                    print(f"    Skipping {filename}: Could not parse correctly.")
                    continue
                
                # Verify the condition is one we are tracking
                if condition not in ['CBX', 'SAL']:
                    print(f"    Skipping {filename}: Unknown condition '{condition}'.")
                    continue
                
                # Load behavioral data
                try:
                    behaviordata = sio.loadmat(matfile)
                    eventlog = np.squeeze(behaviordata['eventlog'])
                    licks = np.squeeze(behaviordata['licks'])
                except Exception as e:
                    print(f"    Error loading {filename}: {e}")
                    continue
                
                # Extract cues
                sucrosecues, nosucrosecues, totalcues = extract_cues(eventlog)
                
                # Calculate delta lick rates for CS+ (Cue and Reward periods)
                cs_plus_cue_rates, cs_plus_cue_avg = calculate_delta_lickrate(
                    sucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000))
                cs_plus_reward_rates, cs_plus_reward_avg = calculate_delta_lickrate(
                    sucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(3000, 6000))
                    
                # Calculate delta lick rates for CS- (Cue and Omission periods)
                cs_minus_cue_rates, cs_minus_cue_avg = calculate_delta_lickrate(
                    nosucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000))
                cs_minus_omission_rates, cs_minus_omission_avg = calculate_delta_lickrate(
                    nosucrosecues, licks, baseline_offset=(-3000, -1), period_offset=(3000, 6000))
                
                # Calculate Discrimination (auROC)
                # FIX: Explicitly passing phase=phase here!
                auroc_norm, auroc_pval = calculate_auROC(
                    cs_plus_cue_rates, cs_minus_cue_rates, phase=phase, offset_to_zero=True)
                
                # Calculate session length
                session_len = find_session_length(eventlog)
                
                # Store all extracted information in the dictionary
                data_dict[phase][condition][session_id][animal_id] = {
                    'file': filename,
                    'Session Length': session_len,
                    'CS+': {
                        'trial_delta_rates': cs_plus_cue_rates.tolist(),
                        'trial_reward_rates': cs_plus_reward_rates.tolist(),
                        'avg_delta_cue': cs_plus_cue_avg,
                        'avg_delta_reward': cs_plus_reward_avg,
                        'total_cues': len(sucrosecues)
                    },
                    'CS-': {
                        'trial_delta_rates': cs_minus_cue_rates.tolist(),
                        'trial_omission_rates': cs_minus_omission_rates.tolist(),
                        'avg_delta_cue': cs_minus_cue_avg,
                        'avg_delta_omission': cs_minus_omission_avg,
                        'total_cues': len(nosucrosecues)
                    },
                    'Discrimination': {
                        'auROC_norm': auroc_norm,
                        'auROC_pval': auroc_pval
                    }
                }
                
    return data_dict

# --- Example Usage ---
if __name__ == "__main__":
    # Update this to your actual base directory
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    
    # Build the dictionary
    print("Building data dictionary from MATLAB files...")
    cbx_data = build_cbx_data_dictionary(basedir)
    print("\nDictionary build complete!")

In [ ]:
###plotting auROCs across days -- used as a check against Prism and to calculate slopes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def plot_cbx_auroc_learning_curves(data_dict, results_dir, max_days=5):
    """
    Extracts normalized auROC values from the dictionary and plots 
    discrimination learning curves across days for CBX vs SAL.
    """
    print(f"\n===== Generating auROC Learning Curves (Days 1-{max_days}) =====")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Flatten the dictionary data into a list for pandas
    plot_data = []
    
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
            
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
                
            for session_id, session_data in data_dict[phase][condition].items():
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                    
                if day_num > max_days:
                    continue
                
                for animal_id, animal_data in session_data.items():
                    if 'Discrimination' in animal_data and 'auROC_norm' in animal_data['Discrimination']:
                        auroc = animal_data['Discrimination']['auROC_norm']
                        
                        # Only append if we have a valid number (skip NaNs from missing trials)
                        if not np.isnan(auroc):
                            plot_data.append({
                                'Phase': phase,
                                'Condition': condition,
                                'Day': day_num,
                                'Animal': animal_id,
                                'auROC': auroc
                            })
                            
    if not plot_data:
        print("No auROC data found in the dictionary!")
        return None
        
    df = pd.DataFrame(plot_data)
    
    # Save the raw auROC data to a CSV for your records
    df.to_csv(results_dir / 'CBX_auROC_Long_Format.csv', index=False)
    
    # 2. Plotting
    colors = {'CBX': 'seagreen', 'SAL': 'black'}
    
    for phase in df['Phase'].unique():
        phase_data = df[df['Phase'] == phase]
        
        # Figure out the actual number of days we have data for
        valid_days = sorted(phase_data['Day'].unique())
        if not valid_days:
            continue
            
        plt.figure(figsize=(8, 6))
        
        for condition in ['CBX', 'SAL']:
            cond_data = phase_data[phase_data['Condition'] == condition]
            if cond_data.empty:
                continue
                
            # Calculate daily means and SEMs
            daily_means = []
            daily_sems = []
            days_plotted = []
            
            for day in valid_days:
                day_vals = cond_data[cond_data['Day'] == day]['auROC']
                if not day_vals.empty:
                    days_plotted.append(day)
                    daily_means.append(day_vals.mean())
                    daily_sems.append(stats.sem(day_vals) if len(day_vals) > 1 else 0)
            
            # Plot the line with error bars
            plt.errorbar(days_plotted, daily_means, yerr=daily_sems,
                         label=f"{condition} (n={len(cond_data['Animal'].unique())})",
                         color=colors[condition], marker='o', markersize=8, 
                         capsize=5, linewidth=2, capthick=2)
            
        # Formatting the plot
        plt.title(f'{phase} Phase - Discrimination Learning (auROC)', fontsize=14, pad=15)
        plt.xlabel('Day', fontsize=12)
        plt.ylabel('Normalized auROC', fontsize=12)
        
        # Set x-ticks strictly to the valid days
        plt.xticks(valid_days, [f'Day {d}' for d in valid_days])
        
        # auROC bounds are -1 to 1. 0 is chance.
        plt.ylim(-1.05, 1.05)
        plt.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Chance (0)')
        
        # Add a subtle grid
        plt.grid(True, linestyle=':', alpha=0.6)
        
        # Place legend
        plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='black')
        
        # Remove top and right borders for a cleaner look
        ax = plt.gca()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Save figure
        save_path = results_dir / f'{phase.lower()}_cbx_auroc_learning_curve.pdf'
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        
    print(f"auROC learning curves saved to {results_dir}")
    return df

# --- Execution Example ---
if __name__ == "__main__":
    # Assuming 'cbx_data' is your populated dictionary from the builder step
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Visualizations'
    
    auroc_df = plot_cbx_auroc_learning_curves(cbx_data, results_dir, max_days=9)

In [ ]:
##slope of the auROC curves

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

def calculate_cbx_auroc_slopes(data_dict, max_days=5):
    """
    Calculate learning slopes for each animal's normalized auROC across a specified learning window.
    """
    slopes_data = {
        'Acquisition': {'CBX': {}, 'SAL': {}},
        'Reversal': {'CBX': {}, 'SAL': {}}
    }
    
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
            
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
                
            # Track animals and their auROC data across days
            animal_daily_data = {}
            
            for session_id in data_dict[phase][condition]:
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                
                # Only include days within our target learning window
                if day_num > max_days:
                    continue
                    
                for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                    if 'Discrimination' in animal_data and 'auROC_norm' in animal_data['Discrimination']:
                        auroc = animal_data['Discrimination']['auROC_norm']
                        
                        if not np.isnan(auroc):
                            if animal_id not in animal_daily_data:
                                animal_daily_data[animal_id] = {'days': [], 'aurocs': []}
                                
                            animal_daily_data[animal_id]['days'].append(day_num)
                            animal_daily_data[animal_id]['aurocs'].append(auroc)
        
            # Calculate linear regression slopes for each animal
            for animal_id, data in animal_daily_data.items():
                # Only calculate slopes if we have at least 3 days of data to fit a line
                if len(data['days']) >= 3:
                    slope, intercept, r_value, p_value, std_err = stats.linregress(
                        data['days'], data['aurocs']
                    )
                    slopes_data[phase][condition][animal_id] = {
                        'slope': slope,
                        'r_squared': r_value**2,
                        'days': data['days'],
                        'aurocs': data['aurocs']
                    }
                    
    return slopes_data

def run_cbx_auroc_slope_statistics(slopes_data):
    """
    Statistical analysis of auROC learning slopes between CBX and SAL.
    """
    stats_results = {'Acquisition': {}, 'Reversal': {}}
    
    for phase in ['Acquisition', 'Reversal']:
        if not slopes_data[phase]['CBX'] or not slopes_data[phase]['SAL']:
            continue
            
        cbx_slopes = np.array([data['slope'] for data in slopes_data[phase]['CBX'].values()])
        sal_slopes = np.array([data['slope'] for data in slopes_data[phase]['SAL'].values()])
        
        if len(cbx_slopes) == 0 or len(sal_slopes) == 0:
            continue
        
        # Assumptions tests
        _, cbx_norm_p = stats.shapiro(cbx_slopes) if len(cbx_slopes) >= 3 else (None, 1.0)
        _, sal_norm_p = stats.shapiro(sal_slopes) if len(sal_slopes) >= 3 else (None, 1.0)
        _, levene_p = stats.levene(cbx_slopes, sal_slopes) if len(cbx_slopes) > 1 and len(sal_slopes) > 1 else (None, 1.0)
        
        # Select test
        if cbx_norm_p > 0.05 and sal_norm_p > 0.05 and levene_p > 0.05:
            t_stat, p_val = stats.ttest_ind(cbx_slopes, sal_slopes)
            test_used = "Independent t-test"
        else:
            stat, p_val = stats.mannwhitneyu(cbx_slopes, sal_slopes, alternative='two-sided')
            t_stat = stat
            test_used = "Mann-Whitney U test"
        
        # Effect size (Cohen's d)
        n1, n2 = len(cbx_slopes), len(sal_slopes)
        if n1 > 1 and n2 > 1:
            pooled_sd = np.sqrt(((n1-1)*np.std(cbx_slopes, ddof=1)**2 + (n2-1)*np.std(sal_slopes, ddof=1)**2) / (n1+n2-2))
            cohens_d = (np.mean(cbx_slopes) - np.mean(sal_slopes)) / pooled_sd if pooled_sd > 0 else 0
        else:
            cohens_d = np.nan
        
        stats_results[phase] = {
            'test_used': test_used,
            'statistic': t_stat,
            'p_value': p_val,
            'cohens_d': cohens_d,
            'CBX_n': n1,
            'SAL_n': n2,
            'CBX_mean': np.mean(cbx_slopes),
            'CBX_sem': stats.sem(cbx_slopes) if n1 > 1 else 0,
            'CBX_std': np.std(cbx_slopes, ddof=1) if n1 > 1 else 0,
            'SAL_mean': np.mean(sal_slopes),
            'SAL_sem': stats.sem(sal_slopes) if n2 > 1 else 0,
            'SAL_std': np.std(sal_slopes, ddof=1) if n2 > 1 else 0,
            'normality_cbx_p': cbx_norm_p,
            'normality_sal_p': sal_norm_p,
            'levene_p': levene_p
        }
        
        # Bonferroni correction (2 phases)
        stats_results[phase]['p_value_corrected'] = min(p_val * 2, 1.0)
            
    return stats_results

def analyze_cbx_auroc_learning_slopes(slopes_data, results_dir, max_days=5):
    """
    Analyze and visualize auROC learning slopes.
    """
    stats_results = run_cbx_auroc_slope_statistics(slopes_data)
    
    # Setup visualization (1 row, 2 columns since there is no CS+/CS- split)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
    
    # Determine y-axis limits based on the data
    all_means = []
    all_errs = []
    for c in stats_results.values():
        if c:
            all_means.extend([c['CBX_mean'], c['SAL_mean']])
            all_errs.extend([c['CBX_sem'], c['SAL_sem']])
            
    if all_means:
        max_val = max([abs(m) + e for m, e in zip(all_means, all_errs)])
        y_lim = max(0.1, max_val * 1.3) # Give a nice buffer
    else:
        y_lim = 0.5
        
    def create_bar_plot(ax, stats, title):
        if not stats:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
            ax.axis('off')
            return
            
        bars = ax.bar(['CBX', 'SAL'], 
                [stats['CBX_mean'], stats['SAL_mean']],
                yerr=[stats['CBX_sem'], stats['SAL_sem']],
                capsize=5,
                color=['seagreen', 'black'])
        
        ax.set_title(title, pad=10, fontsize=12)
        ax.set_ylabel('auROC Learning Slope (Δ auROC / day)', fontsize=10)
        ax.set_ylim(-y_lim, y_lim)
        ax.grid(True, linestyle='--', alpha=0.7, axis='y')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        # Add n values
        for idx, bar in enumerate(bars):
            n = stats['CBX_n'] if idx == 0 else stats['SAL_n']
            ax.text(bar.get_x() + bar.get_width()/2, 
                   bar.get_height() + (0.01 if bar.get_height() >= 0 else -0.01),
                   f'n={n}', ha='center', va='bottom' if bar.get_height() >= 0 else 'top',
                   fontsize=10)
        
        # Significance marker
        if stats['p_value'] < 0.05:
            max_height = max(stats['CBX_mean'], stats['SAL_mean'])
            y_pos = max_height + (y_lim * 0.15) if max_height > 0 else max_height - (y_lim * 0.15)
            marker = "***" if stats['p_value'] < 0.001 else "**" if stats['p_value'] < 0.01 else "*"
            ax.text(0.5, y_pos, marker, ha='center', va='bottom', fontsize=14)

    # Apply to axes
    create_bar_plot(ax1, stats_results.get('Acquisition'), f'Acquisition Slopes (Days 1-{max_days})')
    create_bar_plot(ax2, stats_results.get('Reversal'), f'Reversal Slopes (Days 1-{max_days})')
    
    fig.suptitle('Discrimination Learning Slopes (auROC)\nGroup means ± SEM of linear regression slopes', fontsize=14, y=1.02)
    plt.tight_layout()
    
    # Save files
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(results_dir / f'cbx_auroc_learning_slopes_days1to{max_days}.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    
    # --- Write Detailed Stats to File ---
    output_file = results_dir / 'cbx_auroc_slope_statistics.txt'
    with open(output_file, 'w') as f:
        f.write(f"Detailed Statistical Analysis of auROC Learning Slopes (Days 1-{max_days})\n")
        f.write("="*60 + "\n\n")
        
        for phase in ['Acquisition', 'Reversal']:
            if not stats_results.get(phase):
                continue
                
            stats = stats_results[phase]
            f.write(f"\n{phase} Phase Analysis\n")
            f.write("-" * 20 + "\n")
            
            f.write("\nSample Sizes:\n")
            f.write(f"  CBX: n = {stats['CBX_n']}\n")
            f.write(f"  SAL: n = {stats['SAL_n']}\n")
            
            f.write("\nDescriptive Statistics:\n")
            f.write(f"  CBX Mean Slope: {stats['CBX_mean']:.4f} ± {stats['CBX_sem']:.4f} SEM\n")
            f.write(f"  SAL Mean Slope: {stats['SAL_mean']:.4f} ± {stats['SAL_sem']:.4f} SEM\n")
            
            f.write(f"\nStatistical Analysis ({stats['test_used']}):\n")
            f.write(f"  Test Statistic = {stats['statistic']:.3f}\n") 
            f.write(f"  p-value: {stats['p_value']:.4f}\n")
            if not np.isnan(stats['cohens_d']):
                f.write(f"  Cohen's d: {stats['cohens_d']:.3f}\n")
            
            if stats['p_value'] < 0.05:
                direction = "faster" if stats['CBX_mean'] > stats['SAL_mean'] else "slower"
                f.write(f"  * Significant: CBX group showed {direction} discrimination learning than SAL (p < 0.05)\n")
            else:
                f.write("  * No significant difference between groups\n")
            f.write("\n" + "=" * 50 + "\n")
            
    return stats_results

# --- Execution Example ---
if __name__ == "__main__":
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Statistics'
    
    print("\nCalculating auROC learning slopes...")
    auroc_slopes_data = calculate_cbx_auroc_slopes(cbx_data, max_days=9) 
    
    print("\nVisualizing auROC learning slopes and running statistics...")
    auroc_stats_results = analyze_cbx_auroc_learning_slopes(auroc_slopes_data, results_dir, max_days=9)
    print("\nauROC slope analysis complete! Check the results directory.")

In [ ]:
###scatter plot for latecy to acquire and reverse 

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# --- Data Input ---
# One entry per mouse. Latencies = days to criterion
# (first day of a run of >=2 consecutive days with auROC over threshold).
# Use None for reversal when the mouse hasn't finished reversal yet.
data = [
    # CBX
    ("PFC-CBX1", "CBX", 4, 10),
    ("PFC-CBX3", "CBX", 5, 9),
    ("PFC-CBX4", "CBX", 9, 10),
    ("PFC-CBX6", "CBX", 9, 9),
    ("PFC-CBX9", "CBX", 4, 21),
    ("PFC-CBX10", "CBX", 7, 21),
    ("PFC-Can15", "CBX", 7, 8),
    ("PFC-Can17", "CBX", 3, 9),
    ("PFC-Can19", "CBX", 3, 8),
    ("PFC-Can-Gq2", "CBX", 4, 20),
    ("PFC-Can-TDT4", "CBX", 5, 11),
    ("PFC-Can-TDT7", "CBX", 7, 21),
    ("PFC-Can-TDT8", "CBX", 7, 8),
    # saline
    ("PFC-CBX2", "SAL", 6, 4),
    ("PFC-CBX5", "SAL", 4, 8),
    ("PFC-CBX11", "SAL", 4, 14),
    ("PFC-CBX12", "SAL", 7, 8),
    ("PFC-CAN13", "SAL", 6, 6),
    ("PFC-CAN14", "SAL", 11, 12),
    ("PFC-CAN18", "SAL", 9, 6),
    ("PFC-CAN20", "SAL", 7, 6),
    ("PFC-CAN-GQ1", "SAL", 3, 10),
    ("PFC-CAN-GQ3", "SAL", 4, 21),
    ("PFC-CAN-TDT3", "SAL", 4, 11),
    ("PFC-CAN-TDT6", "SAL", 5, 12)
]

# --- Plotting ---

def plot_latency_correlations(data, save_path=None, show_plot=True, fit_line='none'):
    """
    Side-by-side scatter of acquisition latency (x) vs reversal latency (y),
    one panel per group.

    Each panel shows:
      - the y = x identity diagonal (above it = reversal slower than acquisition)
      - Spearman rho and Pearson r (within-group correlation)
      - that group's difference-score summary (mean rev-acq, and above/on/below counts)
    Below the panels: between-group comparison of difference scores (rev - acq),
    via Mann-Whitney (robust) and t-test.

    fit_line : 'none' (default) | 'theilsen' | 'ols'
        Optional within-group trend line. Left off by default so the diagonal
        is the only reference. Use 'theilsen' if you want a line consistent
        with the rank-based rho; 'ols' is the classic least-squares (Pearson) fit.
    """
    group_style = {
        'SAL': {'color': 'black',    'label': 'Saline'},
        'CBX': {'color': 'seagreen', 'label': 'CBX'},
    }

    # Split by group, keeping acq/rev paired; skip mice with pending reversal
    grouped = {'SAL': {'acq': [], 'rev': []},
               'CBX': {'acq': [], 'rev': []}}
    pending = 0
    for animal, group, acq, rev in data:
        if group not in grouped:
            print(f"[!] Skipping {animal}: unknown group '{group}'")
            continue
        if rev is None:
            pending += 1
            continue
        grouped[group]['acq'].append(acq)
        grouped[group]['rev'].append(rev)
    if pending:
        print(f"[i] {pending} mouse/mice excluded (reversal latency pending).")

    # Per-animal difference scores (rev - acq) = vertical distance from the diagonal
    diffs = {g: np.array(grouped[g]['rev']) - np.array(grouped[g]['acq'])
             for g in ['SAL', 'CBX']}

    # Shared, square axes so the diagonal reads as a true 45 degrees
    all_vals = [v for g in grouped.values() for k in ('acq', 'rev') for v in g[k]]
    max_val = max(all_vals) if all_vals else 21
    top_tick = int(np.ceil(max_val / 7) * 7)   # round up to a multiple of 7 (21 now)
    ticks = np.arange(0, top_tick + 1, 7)       # [0, 7, 14, 21]
    lo, hi = 0, top_tick + 1                     # start at origin, 1 unit headroom

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharex=True, sharey=True)
    fig.suptitle('Acquisition vs. Reversal Latency', fontsize=16, y=0.99)

    for ax, group in zip(axes, ['SAL', 'CBX']):
        style = group_style[group]
        x = np.array(grouped[group]['acq'], dtype=float)
        y = np.array(grouped[group]['rev'], dtype=float)
        n = len(x)
        d = diffs[group]

        # y = x identity diagonal
        ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=1.5,
                alpha=0.7, zorder=1)

        # Scatter (50% opacity for overlap)
        ax.scatter(x, y, color=style['color'], alpha=0.5, s=70,
                   edgecolors='none', zorder=3)

        # Within-group correlation text
        if n >= 3:
            rho, p_rho = stats.spearmanr(x, y)
            r, p_r = stats.pearsonr(x, y)
            corr_txt = (f"Spearman \u03c1 = {rho:.3f}, p = {p_rho:.3f}\n"
                        f"Pearson r = {r:.3f}, p = {p_r:.3f}\n")
        else:
            corr_txt = "(need \u22653 for correlation)\n"

        # Optional within-group fit line (off by default)
        if fit_line == 'ols' and n >= 3:
            slope, intercept = np.polyfit(x, y, 1)
            ax.plot([lo, hi], [slope * lo + intercept, slope * hi + intercept],
                    color=style['color'], lw=2, alpha=0.9, zorder=2)
        elif fit_line == 'theilsen' and n >= 3:
            slope, intercept, _, _ = stats.theilslopes(y, x)
            ax.plot([lo, hi], [slope * lo + intercept, slope * hi + intercept],
                    color=style['color'], lw=2, alpha=0.9, zorder=2)

        # Difference-score summary (distance from the diagonal)
        n_above = int(np.sum(d > 0))
        n_below = int(np.sum(d < 0))
        n_on = int(np.sum(d == 0))
        diff_txt = (f"n = {n}\n"
                    f"mean(rev\u2212acq) = {np.mean(d):+.2f}\n"
                    f"above/on/below = {n_above}/{n_on}/{n_below}")

        ax.text(0.96, 0.04, corr_txt + diff_txt, transform=ax.transAxes,
                fontsize=10.5, va='bottom', ha='right',
                bbox=dict(boxstyle='round', facecolor='white',
                          edgecolor='lightgray', alpha=0.85))

        ax.set_title(style['label'], fontsize=14)
        ax.set_xlabel('Acquisition Latency (days to criterion)', fontsize=12)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)
        ax.set_aspect('equal', adjustable='box')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, linestyle=':', alpha=0.4)
        

    axes[0].set_ylabel('Reversal Latency (days to criterion)', fontsize=12)

    # --- Between-group comparison of difference scores ---
    sal_d, cbx_d = diffs['SAL'], diffs['CBX']
    if len(sal_d) >= 2 and len(cbx_d) >= 2:
        t, p_t = stats.ttest_ind(cbx_d, sal_d)
        u, p_u = stats.mannwhitneyu(cbx_d, sal_d, alternative='two-sided')
        caption = (f"Difference score (reversal \u2212 acquisition), CBX vs SAL:   "
                   f"Mann\u2013Whitney p = {p_u:.3f}   |   t-test p = {p_t:.3f}   "
                   f"(CBX mean {np.mean(cbx_d):+.2f} vs SAL {np.mean(sal_d):+.2f} days)")
        fig.text(0.5, 0.02, caption, ha='center', fontsize=11)

        print("\n--- Difference-score comparison (rev - acq) ---")
        print(f"CBX: mean {np.mean(cbx_d):+.2f} days  (n={len(cbx_d)})")
        print(f"SAL: mean {np.mean(sal_d):+.2f} days  (n={len(sal_d)})")
        print(f"Mann-Whitney U={u:.0f}, p={p_u:.3f}   |   t-test t={t:.2f}, p={p_t:.3f}")

    plt.tight_layout(rect=[0, 0.045, 1, 1])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Saved figure to {save_path}")
    if show_plot:
        plt.show()
    else:
        plt.close(fig)


plot_latency_correlations(data, save_path='latency_correlation.pdf',
                          show_plot=True, fit_line='none')

In [ ]:
####generating the lick plots across time

In [ ]:
import os
import numpy as np
import scipy.io as sio
import scipy.stats as stats
import matplotlib.pyplot as plt
from pathlib import Path

# --- Helper Functions ---

def extract_cues(eventlog):
    """Extract CS+ and CS- timestamps from the event log."""
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    return sucrosecues, nosucrosecues

def process_lick_timeseries(licks, cue_times, bin_size=100, pre_cue_time=3000, post_cue_time=6000):
    """
    Process lick timestamps into TRUE DELTA binned time series.
    Calculates baseline Hz per trial and subtracts it from all bins in that trial.
    """
    total_time = pre_cue_time + post_cue_time
    num_bins = int(total_time / bin_size)
    bin_edges = np.linspace(-pre_cue_time, post_cue_time, num_bins + 1)
    bin_centers = bin_edges[:-1] + bin_size/2
    
    all_trial_bins = np.zeros((len(cue_times), num_bins))
    
    for i, cue_time in enumerate(cue_times):
        start_time = cue_time - pre_cue_time
        end_time = cue_time + post_cue_time
        
        # 1. Get ALL licks within this 9s time window
        trial_licks = licks[(licks >= start_time) & (licks < end_time)]
        aligned_licks = trial_licks - cue_time
        
        # 2. Bin the licks into 100ms chunks and convert to Hz
        hist, _ = np.histogram(aligned_licks, bins=bin_edges)
        raw_lick_rate = hist * (1000 / bin_size)
        
        # 3. Calculate the Baseline Rate (Hz) for this specific trial (-3s to 0s)
        baseline_licks = licks[(licks >= start_time) & (licks < cue_time)]
        baseline_hz = len(baseline_licks) / (pre_cue_time / 1000.0)
        
        # 4. Calculate TRUE DELTA (Raw Hz - Baseline Hz)
        delta_lick_rate = raw_lick_rate - baseline_hz
        
        all_trial_bins[i, :] = delta_lick_rate
    
    return {
        'bin_centers': bin_centers,
        'lick_rates': all_trial_bins
    }

# --- Time-Resolved Dictionary Builder ---

def build_cbx_time_resolved_dictionary(basedir, bin_size=100, pre_cue_time=3000, post_cue_time=12000):
    """
    Build dictionary containing binned DELTA licking data for all phases and sessions.
    """
    time_resolved_data = {
        'Acquisition': {'CBX': {}, 'SAL': {}},
        'Reversal': {'CBX': {}, 'SAL': {}}
    }
    
    basedir_path = Path(basedir)
    
    for phase in ['Acquisition', 'Reversal']:
        print(f"\nProcessing {phase} phase for time-series data...")
        phase_dir = basedir_path / phase
        
        if not phase_dir.exists():
            continue
            
        for session_dir in phase_dir.iterdir():
            if not session_dir.is_dir() or session_dir.name == 'Output CSVs':
                continue
                
            session_id = session_dir.name
            print(f"  Processing session: {session_id}")
            
            # Initialize session structure
            if session_id not in time_resolved_data[phase]['CBX']:
                time_resolved_data[phase]['CBX'][session_id] = {'CS+': {}, 'CS-': {}}
            if session_id not in time_resolved_data[phase]['SAL']:
                time_resolved_data[phase]['SAL'][session_id] = {'CS+': {}, 'CS-': {}}
                
            for matfile in session_dir.glob('*.mat'):
                filename = matfile.name
                clean_name = filename.replace('.mat', '')
                parts = clean_name.split('_')
                
                try:
                    session_idx = parts.index(session_id)
                    animal_id = '_'.join(parts[:session_idx])
                    condition = parts[session_idx + 1].upper()
                except (ValueError, IndexError):
                    continue
                
                if condition not in ['CBX', 'SAL']:
                    continue
                
                # Load and extract
                try:
                    mat_data = sio.loadmat(matfile)
                    eventlog = np.squeeze(mat_data['eventlog'])
                    licks = np.squeeze(mat_data['licks'])
                except Exception as e:
                    print(f"    Error loading {filename}: {e}")
                    continue
                
                cs_plus_cues, cs_minus_cues = extract_cues(eventlog)
                
                # Process timeseries (Calculating DELTA)
                cs_plus_data = process_lick_timeseries(licks, cs_plus_cues, bin_size, pre_cue_time, post_cue_time)
                cs_minus_data = process_lick_timeseries(licks, cs_minus_cues, bin_size, pre_cue_time, post_cue_time)
                
                # Store
                time_resolved_data[phase][condition][session_id]['CS+'][animal_id] = {
                    'bin_centers': cs_plus_data['bin_centers'],
                    'lick_rates': cs_plus_data['lick_rates']
                }
                time_resolved_data[phase][condition][session_id]['CS-'][animal_id] = {
                    'bin_centers': cs_minus_data['bin_centers'],
                    'lick_rates': cs_minus_data['lick_rates']
                }
                
    time_resolved_data['metadata'] = {
        'bin_size': bin_size, 'pre_cue_time': pre_cue_time, 'post_cue_time': post_cue_time
    }
    return time_resolved_data

def calculate_average_lick_rates(time_resolved_data):
    """Calculate mean and SEM for delta lick rates across animals for each session."""
    results = {}
    phases = [p for p in time_resolved_data.keys() if p != 'metadata']
    
    for phase in phases:
        results[phase] = {}
        for condition in ['CBX', 'SAL']:
            results[phase][condition] = {}
            for session_id in time_resolved_data[phase][condition]:
                results[phase][condition][session_id] = {}
                
                for cue_type in ['CS+', 'CS-']:
                    animals_data = time_resolved_data[phase][condition][session_id][cue_type]
                    if not animals_data:
                        continue
                    
                    animals = list(animals_data.keys())
                    bin_centers = animals_data[animals[0]]['bin_centers']
                    all_animal_means = []
                    
                    for animal in animals:
                        animal_mean = np.mean(animals_data[animal]['lick_rates'], axis=0)
                        all_animal_means.append(animal_mean)
                    
                    all_animal_means = np.array(all_animal_means)
                    
                    results[phase][condition][session_id][cue_type] = {
                        'bin_centers': bin_centers,
                        'mean': np.mean(all_animal_means, axis=0),
                        'sem': stats.sem(all_animal_means, axis=0),
                        'n_animals': len(animals)
                    }
    return results

# --- Plotting ---

def plot_cbx_lick_rate_timeseries(avg_lick_rates, results_dir=None):
    """Create time-resolved DELTA licking rate plots for each session and cue type."""
    if results_dir:
        results_dir = Path(results_dir)
        results_dir.mkdir(exist_ok=True, parents=True)
    
    colors = {'CBX': 'seagreen', 'SAL': 'black'} 
    phases = list(avg_lick_rates.keys())
    
    for phase in phases:
        # Get all unique sessions in this phase
        sessions = set(list(avg_lick_rates[phase]['CBX'].keys()) + list(avg_lick_rates[phase]['SAL'].keys()))
        
        for session_id in sorted(sessions):
            for cue_type in ['CS+', 'CS-']:
                has_cbx = session_id in avg_lick_rates[phase]['CBX'] and cue_type in avg_lick_rates[phase]['CBX'][session_id]
                has_sal = session_id in avg_lick_rates[phase]['SAL'] and cue_type in avg_lick_rates[phase]['SAL'][session_id]
                
                if not (has_cbx and has_sal):
                    continue
                
                plt.figure(figsize=(10, 6))
                
                for condition in ['CBX', 'SAL']:
                    data = avg_lick_rates[phase][condition][session_id][cue_type]
                    bin_centers_sec = data['bin_centers'] / 1000
                    
                    plt.plot(bin_centers_sec, data['mean'], color=colors[condition], 
                             label=f"{condition} (n={data['n_animals']})")
                    plt.fill_between(bin_centers_sec, 
                                     data['mean'] - data['sem'], 
                                     data['mean'] + data['sem'], 
                                     color=colors[condition], alpha=0.3)
                
                # Vertical lines
                plt.axvline(x=0, color='black', linestyle='--', label='Cue Onset')
                plt.axvline(x=3, color='black', linestyle='--', label='Cue Offset / Reward')
                
                # Horizontal line perfectly at 0 (Baseline)
                plt.axhline(y=0, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
                
                # Customize
                plt.title(f"{phase} {session_id} - {cue_type} Delta Lick Trace", fontsize=14)
                plt.xlabel("Time relative to cue onset (s)", fontsize=12)
                plt.ylabel("Delta Lick Rate (Hz)", fontsize=12)
                plt.legend(loc='upper right')
                
                # Adjusted limits for Delta plotting (allows dipping below 0)
                plt.xlim(-3, 12)
                plt.ylim(-2, 10)  
                plt.yticks([-2, 0, 2, 4, 6, 8, 10])
                
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                
                if results_dir:
                    save_path = results_dir / f"{phase}_{session_id}_{cue_type.replace('+', 'plus')}_DeltaLickTrace.pdf"
                    plt.savefig(save_path, bbox_inches='tight', dpi=300)
                plt.show()

# --- Execution ---
if __name__ == "__main__":
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Time-Resolved-Licking-Delta'
    
    print("Extracting trial-by-trial baseline subtracted delta rates...")
    time_resolved_data = build_cbx_time_resolved_dictionary(basedir)
    
    print("Calculating group averages...")
    avg_lick_rates = calculate_average_lick_rates(time_resolved_data)
    
    print("Plotting traces...")
    plot_cbx_lick_rate_timeseries(avg_lick_rates, results_dir)
    print("Delta time-resolved plots complete!")

In [ ]:
###looking at raw lick counts across sessions

In [ ]:
import os
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- Helper Functions ---

def extract_cues(eventlog):
    """Extract CS+ and CS- timestamps from the event log."""
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    return sucrosecues, nosucrosecues

def get_relative_trial_licks(cues, licks, pre_window=3000, post_window=6000):
    """
    Extract lick timestamps relative to cue onset for each trial.
    Returns a list of arrays (one array per trial containing lick times in seconds).
    """
    trial_licks_list = []
    
    for cue in cues:
        # Get absolute lick times within this trial's window (-3s to +6s)
        window_licks = licks[(licks > cue - pre_window) & (licks <= cue + post_window)]
        
        # Align to cue onset (0) and convert to seconds for plotting
        rel_licks_sec = (window_licks - cue) / 1000.0
        trial_licks_list.append(rel_licks_sec)
        
    return trial_licks_list

def count_window_licks(trial_licks_list, start_sec, end_sec):
    """Count total licks across all trials that fall within a specific time window."""
    total_count = 0
    for trial in trial_licks_list:
        total_count += np.count_nonzero((trial > start_sec) & (trial <= end_sec))
    return total_count

# --- 1. Raw Lick Dictionary Builder ---

def build_raw_lick_dictionary(basedir):
    """
    Build dictionary containing raw lick counts and trial-by-trial timestamps.
    """
    raw_data_dict = {
        'Acquisition': {'CBX': {}, 'SAL': {}},
        'Reversal': {'CBX': {}, 'SAL': {}}
    }
    
    basedir_path = Path(basedir)
    
    for phase in ['Acquisition', 'Reversal']:
        print(f"\nProcessing {phase} phase for raw licks...")
        phase_dir = basedir_path / phase
        
        if not phase_dir.exists():
            continue
            
        for session_dir in phase_dir.iterdir():
            if not session_dir.is_dir() or session_dir.name == 'Output CSVs':
                continue
                
            session_id = session_dir.name
            print(f"  Processing session: {session_id}")
            
            if session_id not in raw_data_dict[phase]['CBX']:
                raw_data_dict[phase]['CBX'][session_id] = {}
            if session_id not in raw_data_dict[phase]['SAL']:
                raw_data_dict[phase]['SAL'][session_id] = {}
                
            for matfile in session_dir.glob('*.mat'):
                filename = matfile.name
                
                # --- Bulletproof Filename Parsing ---
                clean_name = filename.replace('.mat', '')
                parts = clean_name.split('_')
                
                try:
                    session_idx = parts.index(session_id)
                    animal_id = '_'.join(parts[:session_idx])
                    condition = parts[session_idx + 1].upper()
                except (ValueError, IndexError):
                    continue
                
                if condition not in ['CBX', 'SAL']:
                    continue
                
                try:
                    behaviordata = sio.loadmat(matfile)
                    eventlog = np.squeeze(behaviordata['eventlog'])
                    licks = np.squeeze(behaviordata['licks'])
                except Exception as e:
                    print(f"    Error loading {filename}: {e}")
                    continue
                
                # Extract Cues and Total Licks
                cs_plus_cues, cs_minus_cues = extract_cues(eventlog)
                total_session_licks = len(licks) if licks.size > 0 else 0
                
                # Get relative timestamps for raster plots (-3s to +6s)
                cs_plus_trial_licks = get_relative_trial_licks(cs_plus_cues, licks)
                cs_minus_trial_licks = get_relative_trial_licks(cs_minus_cues, licks)
                
                # Count licks in specific windows
                # Baseline (-3s to 0s)
                cs_plus_baseline_count = count_window_licks(cs_plus_trial_licks, start_sec=-3, end_sec=0)
                cs_minus_baseline_count = count_window_licks(cs_minus_trial_licks, start_sec=-3, end_sec=0)
                
                # Cue (0s to 3s)
                cs_plus_cue_count = count_window_licks(cs_plus_trial_licks, start_sec=0, end_sec=3)
                cs_minus_cue_count = count_window_licks(cs_minus_trial_licks, start_sec=0, end_sec=3)
                
                # Post-Cue / Reward / Omission (3s to 6s)
                cs_plus_reward_count = count_window_licks(cs_plus_trial_licks, start_sec=3, end_sec=6)
                cs_minus_omission_count = count_window_licks(cs_minus_trial_licks, start_sec=3, end_sec=6)
                
                # Calculate ITI licks via subtraction 
                # (Only subtracting the active cue/post-cue windows, leaving baseline in the ITI pool)
                total_task_licks = (cs_plus_cue_count + cs_plus_reward_count + 
                                    cs_minus_cue_count + cs_minus_omission_count)
                iti_licks = total_session_licks - total_task_licks
                
                # Store in dictionary
                raw_data_dict[phase][condition][session_id][animal_id] = {
                    'Total_Session_Licks': total_session_licks,
                    'CS_Plus_Baseline_Licks': cs_plus_baseline_count,
                    'CS_Plus_Cue_Licks': cs_plus_cue_count,
                    'CS_Plus_Reward_Licks': cs_plus_reward_count,
                    'CS_Minus_Baseline_Licks': cs_minus_baseline_count,
                    'CS_Minus_Cue_Licks': cs_minus_cue_count,
                    'CS_Minus_Omission_Licks': cs_minus_omission_count,
                    'ITI_Licks': iti_licks,
                    # Arrays kept for the raster plots
                    'Raster_CS_Plus': cs_plus_trial_licks,
                    'Raster_CS_Minus': cs_minus_trial_licks
                }
                
    return raw_data_dict

# --- 2. CSV Exporter ---

def export_raw_licks_to_csv(raw_dict, results_dir):
    """Flatten the summary counts and export to CSV."""
    csv_data = []
    
    for phase in ['Acquisition', 'Reversal']:
        for condition in ['CBX', 'SAL']:
            if condition not in raw_dict[phase]:
                continue
            for session_id in raw_dict[phase][condition]:
                for animal_id, data in raw_dict[phase][condition][session_id].items():
                    csv_data.append({
                        'Phase': phase,
                        'Condition': condition,
                        'Session': session_id,
                        'Animal': animal_id,
                        'Total_Session_Licks': data['Total_Session_Licks'],
                        'CS_Plus_Baseline_Licks': data['CS_Plus_Baseline_Licks'],
                        'CS_Plus_Cue_Licks': data['CS_Plus_Cue_Licks'],
                        'CS_Plus_Reward_Licks': data['CS_Plus_Reward_Licks'],
                        'CS_Minus_Baseline_Licks': data['CS_Minus_Baseline_Licks'],
                        'CS_Minus_Cue_Licks': data['CS_Minus_Cue_Licks'],
                        'CS_Minus_Omission_Licks': data['CS_Minus_Omission_Licks'],
                        'ITI_Licks': data['ITI_Licks']
                    })
                    
    if csv_data:
        df = pd.DataFrame(csv_data)
        out_path = Path(results_dir) / 'CBX_Raw_Lick_Counts_Summary.csv'
        df.to_csv(out_path, index=False)
        print(f"Exported raw lick counts to {out_path}")
        return df
    return None

# --- 3. Raster Plot Generator ---

def plot_lick_rasters(raw_dict, results_dir, target_sessions=None,
                      save_plots=True, show_plots=False):
    """
    Generate side-by-side raster plots for CS+ and CS- per animal per session.

    Parameters
    ----------
    target_sessions : None | str | list[str]
        Which day(s)/session(s) to plot.
          None              -> every session (original behavior)
          'Day3'            -> only that session
          ['Day1', 'Day3']  -> only those sessions
    save_plots : bool
        Save each figure as a PDF in results_dir/Raster_Plots.
    show_plots : bool
        Display each figure instead of closing it silently.
    """
    raster_dir = Path(results_dir) / 'Raster_Plots'
    raster_dir.mkdir(exist_ok=True, parents=True)

    # Normalize the selection into a set (or None for "all")
    if target_sessions is None:
        wanted = None
    elif isinstance(target_sessions, str):
        wanted = {target_sessions}
    else:
        wanted = set(target_sessions)

    # Collect what actually exists so we can warn on typos
    available_sessions = set()
    for phase in ['Acquisition', 'Reversal']:
        for condition in ['CBX', 'SAL']:
            available_sessions.update(raw_dict.get(phase, {}).get(condition, {}).keys())

    if wanted is not None:
        missing = wanted - available_sessions
        if missing:
            print(f"[!] Warning: requested sessions not found: {sorted(missing)}")
            print(f"    Available sessions: {sorted(available_sessions)}")

    plotted = 0
    for phase in ['Acquisition', 'Reversal']:
        for condition in ['CBX', 'SAL']:
            if condition not in raw_dict[phase]:
                continue
            for session_id in raw_dict[phase][condition]:

                # --- Day / session selection ---
                if wanted is not None and session_id not in wanted:
                    continue

                for animal_id, data in raw_dict[phase][condition][session_id].items():
                    cs_plus_trials = data['Raster_CS_Plus']
                    cs_minus_trials = data['Raster_CS_Minus']

                    if not cs_plus_trials and not cs_minus_trials:
                        continue

                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
                    fig.suptitle(f'{animal_id} | {phase} | {session_id} | {condition}', fontsize=16)

                    def format_raster_ax(ax, trial_data, title, cue_color):
                        if trial_data and any(len(t) > 0 for t in trial_data):
                            ax.eventplot(trial_data,
                                         lineoffsets=np.arange(1, len(trial_data) + 1),
                                         linelengths=0.8, color='black', alpha=0.8)
                        ax.set_title(title, fontsize=14)
                        ax.set_xlabel('Time from Cue Onset (s)', fontsize=12)
                        ax.set_xlim(-3, 6)
                        ax.axvline(0, color=cue_color, linestyle='-', linewidth=2, alpha=0.6, label='Cue ON')
                        ax.axvline(3, color='gray', linestyle='--', linewidth=2, alpha=0.6, label='Cue OFF')
                        num_trials = len(trial_data) if trial_data else 30
                        ax.set_ylim(0.5, num_trials + 0.5)
                        ax.legend(loc='upper right')

                    format_raster_ax(ax1, cs_plus_trials, 'CS+ Trials', 'green')
                    ax1.set_ylabel('Trial Number', fontsize=12)
                    format_raster_ax(ax2, cs_minus_trials, 'CS- Trials', 'crimson')

                    # Trial 1 at the top (sharey propagates to both axes)
                    ax1.invert_yaxis()
                    plt.tight_layout()

                    if save_plots:
                        save_name = f"{animal_id}_{session_id}_{phase}_{condition}_raster.pdf"
                        fig.savefig(raster_dir / save_name, bbox_inches='tight', dpi=300)

                    if show_plots:
                        plt.show()
                    else:
                        plt.close(fig)

                    plotted += 1

    summary = f"Done. Generated {plotted} raster figure(s)"
    summary += f" for session(s): {sorted(wanted)}" if wanted else " (all sessions)"
    summary += f". Saved to {raster_dir}" if save_plots else "."
    print(summary)

# --- Execution ---
if __name__ == "__main__":
    # Ensure this path points to the folder CONTAINING 'Acquisition' and 'Reversal'
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Raw_Lick_Analysis'
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Build Dictionary
    raw_lick_dict = build_raw_lick_dictionary(basedir)
    
    # Safety Check
    has_data = any(raw_lick_dict[phase]['CBX'] or raw_lick_dict[phase]['SAL'] for phase in ['Acquisition', 'Reversal'])
    
    if not has_data:
        print("\n[!] ERROR: No data was extracted. Please check your basedir path!")
    else:
        # 2. Export CSV
        export_raw_licks_to_csv(raw_lick_dict, results_dir)
        
        # 3. Plot Rasters
        print("\nGenerating Raster Plots...")
    # None -> all days | 'Day3' -> one day | ['Day1','Day5'] -> several
    plot_lick_rasters(raw_lick_dict, results_dir, target_sessions='REV-D7')
    print("\nRaw lick analysis complete!")

In [ ]:
###representative single-animal lick rasters + delta lick traces (early/late acquisition and reversal)
# Paste this cell AFTER the time-resolved licking cell and the raw lick / raster cell.
# Traces come from `time_resolved_data` (time-resolved licking cell).
# Rasters load only the picked sessions' .mat files, so they span the same -3 to +12 s window as the traces
# (raw_lick_dict from the raster cell only keeps licks out to +6 s).

import numpy as np
import scipy.io as sio
import scipy.stats as stats
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# YOUR PICKS -- edit these
#   'Stage': ('Animal ID', 'Session folder name')
#   Session names must match the folder names, e.g. 'PC-D1', 'REV-D9'.
# ============================================================
representative_picks = {
    'SAL': {
        'Early Acquisition': ('PFC-CAN20', 'PC-D1'),
        'Late Acquisition':  ('PFC-CAN20', 'PC-D9'),
        'Early Reversal':    ('PFC-CAN18', 'REV-D1'),
        'Late Reversal':     ('PFC-CAN18', 'REV-D9'),
    },
    'CBX': {
        'Early Acquisition': ('PFC-CAN15', 'PC-D1'),
        'Late Acquisition':  ('PFC-CAN15', 'PC-D9'),
        'Early Reversal':    ('PFC-CAN17', 'REV-D1'),
        'Late Reversal':     ('PFC-CAN17', 'REV-D9'),
    },
}

# --- Helper Functions ---

def get_representative_trace(time_resolved_data, condition, animal_id, session_id):
    """
    Pull one animal's CS+ and CS- delta lick traces for one session.
    Returns the mean and SEM across that animal's trials for each cue.
    """
    # Find which phase this session lives in (PC-* -> Acquisition, REV-* -> Reversal)
    phase = next((p for p in ['Acquisition', 'Reversal']
                  if session_id in time_resolved_data.get(p, {}).get(condition, {})), None)
    if phase is None:
        raise ValueError(f"Session '{session_id}' not found for {condition}. Check the folder name.")

    session = time_resolved_data[phase][condition][session_id]

    # Case-insensitive animal lookup (so 'PFC-Can15' still finds 'PFC-CAN15')
    available = sorted(session['CS+'].keys())
    match = [a for a in available if a.upper() == animal_id.upper()]
    if not match:
        raise ValueError(f"'{animal_id}' has no {condition} data on {session_id}.\n"
                         f"    {condition} animals on {session_id}: {available}")
    animal = match[0]

    trace = {'phase': phase, 'animal': animal}
    for cue_type in ['CS+', 'CS-']:
        cue_data = session[cue_type][animal]
        rates = cue_data['lick_rates']           # trials x bins (delta Hz)
        trace['bin_centers_sec'] = cue_data['bin_centers'] / 1000
        trace[cue_type] = {
            'mean': np.mean(rates, axis=0),
            'sem': stats.sem(rates, axis=0),
            'n_trials': rates.shape[0]
        }
    return trace


def load_representative_raster(basedir, phase, condition, animal, session_id,
                               pre_cue_time=3000, post_cue_time=12000):
    """
    Load one animal's .mat file for one session and return cue-aligned lick times (s)
    for every CS+ and CS- trial, using the same window as the delta traces.
    """
    session_dir = Path(basedir) / phase / session_id

    for matfile in session_dir.glob('*.mat'):
        # Same filename parsing as the dictionary builders
        parts = matfile.name.replace('.mat', '').split('_')
        try:
            session_idx = parts.index(session_id)
            file_animal = '_'.join(parts[:session_idx])
            file_condition = parts[session_idx + 1].upper()
        except (ValueError, IndexError):
            continue
        if file_animal.upper() != animal.upper() or file_condition != condition:
            continue

        mat_data = sio.loadmat(matfile)
        eventlog = np.squeeze(mat_data['eventlog'])
        licks = np.atleast_1d(np.squeeze(mat_data['licks']))

        cues = {'CS+': eventlog[eventlog[:, 0] == 5, 1],    # same codes as extract_cues
                'CS-': eventlog[eventlog[:, 0] == 6, 1]}

        rasters = {}
        for cue_type, cue_times in cues.items():
            rasters[cue_type] = [
                (licks[(licks >= cue - pre_cue_time) & (licks < cue + post_cue_time)] - cue) / 1000.0
                for cue in cue_times
            ]
        return rasters

    raise FileNotFoundError(f"No .mat file for {animal} ({condition}) in {session_dir}")


def draw_raster(ax, rasters, cue_colors):
    """Stack CS+ trials (top) above CS- trials (bottom), trial 1 at the top of each block."""
    y = 0
    block_centers = []
    for cue_type in ['CS+', 'CS-']:
        trials = rasters[cue_type]
        if trials:
            ax.eventplot(trials, lineoffsets=np.arange(y + 1, y + len(trials) + 1),
                         linelengths=0.8, linewidths=0.6, colors=cue_colors[cue_type])
        block_centers.append(y + (len(trials) + 1) / 2)
        y += len(trials)
        if cue_type == 'CS+':
            ax.axhline(y + 0.5, color='gray', linewidth=0.8)   # divider between CS+ and CS- trials

    ax.set_ylim(y + 0.5, 0.5)   # inverted so trial 1 sits at the top
    ax.set_yticks(block_centers)
    ax.set_yticklabels(['CS+', 'CS-'])
    for tick_label, cue_type in zip(ax.get_yticklabels(), ['CS+', 'CS-']):
        tick_label.set_color(cue_colors[cue_type])

# --- Plotting ---

def plot_representative_lick_traces(time_resolved_data, picks, basedir, results_dir=None,
                                    y_limits=(-2, 10),
                                    save_name='Representative_DeltaLickTraces.pdf'):
    """
    For each group: a row of rasters above a row of delta lick traces.
    Columns = Early/Late Acquisition, Early/Late Reversal.
    Each trace panel overlays the chosen animal's CS+ and CS- traces (mean +/- SEM across trials).
    """
    stages = ['Early Acquisition', 'Late Acquisition', 'Early Reversal', 'Late Reversal']
    groups = list(picks.keys())
    cue_colors = {'CS+': 'green', 'CS-': 'crimson'}       # same cue colors as the raster plots
    group_colors = {'CBX': 'seagreen', 'SAL': 'black'}    # same group colors as the group traces

    meta = time_resolved_data.get('metadata', {})
    pre_ms = meta.get('pre_cue_time', 3000)
    post_ms = meta.get('post_cue_time', 12000)

    fig, axes = plt.subplots(2 * len(groups), len(stages),
                             figsize=(4.5 * len(stages), 7.5 * len(groups)),
                             sharex=True, squeeze=False, constrained_layout=True,
                             gridspec_kw={'height_ratios': [1, 1.1] * len(groups)})

    print(f"{'Group':<6}{'Stage':<20}{'Animal':<16}{'Session':<10}{'CS+ trials':>11}{'CS- trials':>11}")

    for g, group in enumerate(groups):
        for col, stage in enumerate(stages):
            raster_ax = axes[2 * g, col]
            trace_ax = axes[2 * g + 1, col]

            if stage not in picks[group]:
                for ax in (raster_ax, trace_ax):
                    ax.text(0.5, 0.5, 'not set', ha='center', va='center',
                            transform=ax.transAxes, color='gray')
                continue

            animal_id, session_id = picks[group][stage]
            trace = get_representative_trace(time_resolved_data, group, animal_id, session_id)
            rasters = load_representative_raster(basedir, trace['phase'], group, trace['animal'],
                                                 session_id, pre_ms, post_ms)
            t = trace['bin_centers_sec']

            print(f"{group:<6}{stage:<20}{trace['animal']:<16}{session_id:<10}"
                  f"{trace['CS+']['n_trials']:>11}{trace['CS-']['n_trials']:>11}")

            # --- Raster (top) ---
            draw_raster(raster_ax, rasters, cue_colors)
            raster_ax.set_title(f"{stage}\n{trace['animal']} | {session_id}", fontsize=11)

            # --- Delta trace (bottom) ---
            for cue_type in ['CS+', 'CS-']:
                d = trace[cue_type]
                trace_ax.plot(t, d['mean'], color=cue_colors[cue_type], linewidth=1.5,
                              label=f"{cue_type} ({d['n_trials']} trials)")
                trace_ax.fill_between(t, d['mean'] - d['sem'], d['mean'] + d['sem'],
                                      color=cue_colors[cue_type], alpha=0.25, linewidth=0)

                # Flag anything that runs off the shared y-axis
                if np.nanmax(d['mean'] + d['sem']) > y_limits[1] or np.nanmin(d['mean'] - d['sem']) < y_limits[0]:
                    print(f"    [!] {group} {stage} {cue_type} trace exceeds y-limits {y_limits}")

            trace_ax.axhline(y=0, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
            trace_ax.set_ylim(*y_limits)
            trace_ax.set_yticks(np.arange(y_limits[0], y_limits[1] + 1, 2))
            trace_ax.legend(loc='upper right', fontsize=8, frameon=False)
            trace_ax.grid(True, alpha=0.3)

            # Shared styling: cue onset / offset lines, no top/right spines
            for ax in (raster_ax, trace_ax):
                ax.axvline(x=0, color='black', linestyle='--', linewidth=1)   # cue onset
                ax.axvline(x=3, color='black', linestyle='--', linewidth=1)   # cue offset / reward
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

        # Labels: group name on the raster row, units on the trace row; x labels under every trace row
        axes[2 * g, 0].set_ylabel(f"{group}\nTrials", fontsize=12, color=group_colors.get(group, 'black'))
        axes[2 * g + 1, 0].set_ylabel("Delta Lick Rate (Hz)", fontsize=11)
        for ax in axes[2 * g + 1, :]:
            ax.tick_params(labelbottom=True)
            ax.set_xlabel("Time relative to cue onset (s)", fontsize=11)

    axes[0, 0].set_xlim(-pre_ms / 1000, post_ms / 1000)

    fig.suptitle("Representative Lick Rasters and Delta Lick Traces", fontsize=15)

    if results_dir:
        results_dir = Path(results_dir)
        results_dir.mkdir(exist_ok=True, parents=True)
        save_path = results_dir / save_name
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"\nSaved to {save_path}")

    plt.show()
    return fig

# --- Execution ---
if 'time_resolved_data' not in globals():
    raise NameError("`time_resolved_data` not found. Run the time-resolved licking cell first.")

basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
rep_results_dir = Path(basedir) / 'Results' / 'Time-Resolved-Licking-Delta'

# y_limits matches the group delta traces; widen it (e.g. (-2, 12)) if a trace gets flagged as clipped
# (assigned to rep_fig so Jupyter doesn't display the figure a second time)
rep_fig = plot_representative_lick_traces(time_resolved_data, representative_picks, basedir,
                                          rep_results_dir, y_limits=(-2, 10))

In [ ]:
##moving into signal detection theory response type breakdowns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency

def classify_cbx_response(phase, cue_type, is_high_licking):
    """
    Classify response type based on phase, cue type, and licking behavior.
    Threshold for 'is_high_licking' is typically > 1.5 Hz.
    """
    if phase == 'Acquisition':
        if cue_type == 'CS+':
            return 'correct response' if is_high_licking else 'incorrect withholding'
        else:  # CS-
            return 'incorrect response' if is_high_licking else 'correct withholding'
    else:  # Reversal
        if cue_type == 'CS+':
            return 'incorrect response' if is_high_licking else 'correct withholding'
        else:  # CS-
            return 'correct response' if is_high_licking else 'incorrect withholding'

def get_significance_stars(p_val):
    """Convert p-value to significance stars."""
    if np.isnan(p_val): return "N/A"
    if p_val < 0.001: return "***"
    elif p_val < 0.01: return "**"
    elif p_val < 0.05: return "*"
    else: return "ns"

def analyze_and_plot_cbx_response_pie_charts(data_dict, results_dir, max_days=5, threshold=1.5):
    """
    Creates pie charts for response types and runs Chi-Square tests of independence 
    between CBX and SAL within each cue type per day.
    """
    print(f"\n{'='*60}")
    print("Running Categorical SDT Analysis (Pie Charts & Chi-Square)")
    print(f"{'='*60}")
    
    colors_map = {
        'correct response': 'greenyellow',
        'correct withholding': 'greenyellow',
        'incorrect response': 'darkslategrey',
        'incorrect withholding': 'grey'
    }
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    stats_output_file = results_dir / 'CBX_SDT_ChiSquare_Stats.txt'
    
    with open(stats_output_file, 'w') as f:
        f.write("Categorical SDT Analysis: Chi-Square Tests of Independence\n")
        f.write("Comparing CBX vs SAL response distributions within Cue Type per Day.\n")
        f.write("="*70 + "\n\n")
    
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
            
        print(f"\nProcessing {phase} phase...")
        
        # Find all valid sessions up to max_days
        all_sessions = set()
        for cond in ['CBX', 'SAL']:
            if cond in data_dict[phase]:
                all_sessions.update(data_dict[phase][cond].keys())
        
        valid_sessions = []
        for session in all_sessions:
            try:
                day_num = int(session.split('-D')[1])
                if day_num <= max_days:
                    valid_sessions.append((day_num, session))
            except (IndexError, ValueError):
                continue
                
        valid_sessions.sort()
        num_rows = len(valid_sessions)
        
        if num_rows == 0:
            print(f"No valid sessions found for {phase} phase to plot.")
            continue
            
        fig, axes = plt.subplots(num_rows, 2, figsize=(15, 5 * num_rows))
        if num_rows == 1:
            axes = np.expand_dims(axes, axis=0)
            
        fig.suptitle(f'{phase} Phase - SDT Response Types (CBX vs SAL)',
                     fontsize=16, fontweight='bold', y=0.95 + (0.01 * (5 - num_rows)))
        
        with open(stats_output_file, 'a') as f:
            f.write(f"\n{'*'*70}\nPHASE: {phase.upper()}\n{'*'*70}\n")
        
        # --- PROCESS EACH DAY ---
        for row_idx, (day_num, session_id) in enumerate(valid_sessions):
            
            day_data = {'CBX': {'CS+': [], 'CS-': []}, 'SAL': {'CS+': [], 'CS-': []}}
            
            # 1. Gather all data for this day
            for condition in ['CBX', 'SAL']:
                if condition in data_dict[phase] and session_id in data_dict[phase][condition]:
                    for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                        for cue_type in ['CS+', 'CS-']:
                            if cue_type in animal_data and 'trial_delta_rates' in animal_data[cue_type]:
                                for rate in animal_data[cue_type]['trial_delta_rates']:
                                    is_high = rate > threshold
                                    resp = classify_cbx_response(phase, cue_type, is_high)
                                    day_data[condition][cue_type].append(resp)

            # 2. Run Chi-Square Stats for this Day
            day_stats_str = f"\n--- DAY {day_num} ({session_id}) ---\n"
            print(day_stats_str, end="")
            with open(stats_output_file, 'a') as f: f.write(day_stats_str)
            
            p_vals = {'CS+': np.nan, 'CS-': np.nan}
            
            for cue_type in ['CS+', 'CS-']:
                cbx_resps = day_data['CBX'][cue_type]
                sal_resps = day_data['SAL'][cue_type]
                
                if not cbx_resps or not sal_resps:
                    continue
                    
                # Determine the two possible response categories for this cue
                unique_resps = sorted(list(set(cbx_resps + sal_resps)))
                
                # Build the 2x2 Contingency Table
                cbx_counts = [cbx_resps.count(r) for r in unique_resps]
                sal_counts = [sal_resps.count(r) for r in unique_resps]
                contingency_table = [cbx_counts, sal_counts]
                
                try:
                    # Run Chi-Square
                    chi2, p, dof, expected = chi2_contingency(contingency_table)
                    p_vals[cue_type] = p
                    stars = get_significance_stars(p)
                    
                    # Format the output table
                    table_str = f"\n{cue_type} Contingency Table:\n"
                    table_str += f"{'':<10} | {unique_resps[0]:<22} | {unique_resps[1] if len(unique_resps)>1 else 'N/A':<22}\n"
                    table_str += "-"*60 + "\n"
                    table_str += f"{'CBX':<10} | {cbx_counts[0]:<22} | {cbx_counts[1] if len(cbx_counts)>1 else 0:<22}\n"
                    table_str += f"{'SAL':<10} | {sal_counts[0]:<22} | {sal_counts[1] if len(sal_counts)>1 else 0:<22}\n"
                    table_str += f"Results: Chi2 = {chi2:.3f}, p = {p:.4f} {stars}\n"
                    
                    print(table_str, end="")
                    with open(stats_output_file, 'a') as f: f.write(table_str)
                    
                except ValueError:
                    err_str = f"\n{cue_type} Contingency Table: Error running stats (likely due to zero variance).\n"
                    print(err_str, end="")
                    with open(stats_output_file, 'a') as f: f.write(err_str)

            # 3. Generate the Pie Charts
            for cond_idx, condition in enumerate(['CBX', 'SAL']):
                ax = axes[row_idx, cond_idx]
                
                if not day_data[condition]['CS+'] and not day_data[condition]['CS-']:
                    ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
                    ax.axis('off')
                    continue
                
                plot_data = []
                for cue_type in ['CS+', 'CS-']:
                    responses = day_data[condition][cue_type]
                    if not responses: continue
                        
                    unique, counts = np.unique(responses, return_counts=True)
                    # Scale to 50% each so CS+ and CS- share the pie equally
                    props = dict(zip(unique, counts / len(responses) * 50))
                    
                    for resp_type, prop in props.items():
                        plot_data.append({
                            'label': f'{cue_type}\n{resp_type}',
                            'size': prop,
                            'color': colors_map[resp_type]
                        })
                
                if plot_data:
                    sizes = [d['size'] for d in plot_data]
                    labels = [d['label'] for d in plot_data]
                    colors_list = [d['color'] for d in plot_data]
                    
                    wedges, texts, autotexts = ax.pie(
                        sizes, labels=labels, colors=colors_list,
                        autopct='%1.1f%%', startangle=90, labeldistance=1.05
                    )
                    plt.setp(autotexts, weight='bold')
                    plt.setp(texts, size=9)
                    
                ax.set_title(f'Day {day_num} - {condition}', pad=10, fontweight='bold')
                
                # --- ADD SIGNIFICANCE STARS TO THE PLOT ---
                # We place the text slightly to the right of the SAL plot
                if cond_idx == 1: 
                    star_text = "Group Comparison:\n"
                    star_text += f"CS+: {get_significance_stars(p_vals['CS+'])}\n"
                    star_text += f"CS-: {get_significance_stars(p_vals['CS-'])}"
                    
                    ax.text(1.5, 0, star_text, fontsize=12, fontweight='bold', 
                            ha='left', va='center', bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

        # Final Plot Formatting
        plt.tight_layout(rect=[0, 0.03, 0.9, 0.95]) # Leave room on right for the stars box
        save_path = results_dir / f'{phase.lower()}_cbx_response_pie_charts.pdf'
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()

    print(f"\nSDT Chi-Square text file successfully saved to:\n{stats_output_file}")

# --- Execution Example ---
if __name__ == "__main__":
    # Assuming 'cbx_data' is the dictionary you built previously
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Visualizations_and_Stats'
    
    analyze_and_plot_cbx_response_pie_charts(cbx_data, results_dir, max_days=9, threshold=1.5)

In [ ]:
###signal detection theory metrics

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols

def calculate_cbx_sdt_metrics(data_dict, threshold=1.5):
    """
    Calculate Signal Detection Theory metrics for each animal and add them to the dictionary.
    """
    def compute_sdt(cs_plus_rates, cs_minus_rates, phase):
        # Determine hits and false alarms based on delta lick rate threshold
        cs_plus_responses = sum(1 for rate in cs_plus_rates if rate > threshold)
        cs_minus_responses = sum(1 for rate in cs_minus_rates if rate > threshold)
        
        cs_plus_total = len(cs_plus_rates)
        cs_minus_total = len(cs_minus_rates)
        
        if cs_plus_total == 0 or cs_minus_total == 0:
            return None
            
        if phase == 'Acquisition':
            # CS+ is the signal
            hit_rate = cs_plus_responses / cs_plus_total
            fa_rate = cs_minus_responses / cs_minus_total
        else:
            # Reversal: CS- is the signal
            hit_rate = cs_minus_responses / cs_minus_total
            fa_rate = cs_plus_responses / cs_plus_total
            
        # Add small epsilon to avoid infinite z-scores (0 or 1 rates)
        epsilon = 0.001
        hit_rate = np.clip(hit_rate, epsilon, 1 - epsilon)
        fa_rate = np.clip(fa_rate, epsilon, 1 - epsilon)
        
        # Calculate d' and beta
        z_hit = norm.ppf(hit_rate)
        z_fa = norm.ppf(fa_rate)
        
        d_prime = z_hit - z_fa
        beta = np.exp((z_fa**2 - z_hit**2) / 2)
        
        if phase == 'Acquisition':
            return {
                'd_prime': d_prime,
                'beta': beta,
                'cs_plus_correct_rate': hit_rate,
                'cs_minus_correct_rate': 1 - fa_rate
            }
        else:
            return {
                'd_prime': d_prime,
                'beta': beta,
                'cs_plus_correct_rate': 1 - fa_rate,
                'cs_minus_correct_rate': hit_rate
            }

    # Iterate through the dictionary structure
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
            for session_id in data_dict[phase][condition]:
                for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                    
                    if 'CS+' in animal_data and 'CS-' in animal_data:
                        cs_plus_rates = animal_data['CS+']['trial_delta_rates']
                        cs_minus_rates = animal_data['CS-']['trial_delta_rates']
                        
                        sdt_metrics = compute_sdt(cs_plus_rates, cs_minus_rates, phase)
                        
                        if sdt_metrics:
                            data_dict[phase][condition][session_id][animal_id]['sdt_metrics'] = sdt_metrics
                            
    return data_dict

def analyze_cbx_sdt_metrics(data_dict, results_dir, max_days=5):
    """
    Perform Two-Way ANOVA on SDT metrics and save results.
    """
    metrics = ['d_prime', 'beta', 'cs_plus_correct_rate', 'cs_minus_correct_rate']
    metric_labels = ["d'", "beta", 'CS+ Correct Rate', 'CS- Correct Rate']
    
    results_file = results_dir / 'CBX_SDT_Statistical_Analysis.txt'
    
    # Flatten data for stats
    stats_data = []
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
            for session_id in data_dict[phase][condition]:
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                    
                if day_num > max_days:
                    continue
                    
                for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                    if 'sdt_metrics' in animal_data:
                        row = {
                            'Phase': phase,
                            'Condition': condition,
                            'Day': day_num,
                            'Animal': animal_id
                        }
                        for metric in metrics:
                            row[metric] = animal_data['sdt_metrics'][metric]
                        stats_data.append(row)
                        
    df = pd.DataFrame(stats_data)
    if df.empty:
        print("No SDT data available for statistical analysis.")
        return
        
    df.to_csv(results_dir / 'CBX_SDT_Metrics_Data.csv', index=False)

    with open(results_file, 'w', encoding='utf-8') as f:
        for phase in df['Phase'].unique():
            phase_data = df[df['Phase'] == phase].copy()
            
            f.write(f"\n{'='*80}\n")
            f.write(f"ANALYSIS FOR {phase.upper()} PHASE (Days 1-{max_days})\n")
            f.write(f"{'='*80}\n\n")
            
            for metric, metric_label in zip(metrics, metric_labels):
                f.write(f"\n{'-'*40}\n")
                f.write(f"Analysis for {metric_label}\n")
                f.write(f"{'-'*40}\n")
                
                metric_data = phase_data[['Condition', 'Day', 'Animal', metric]].dropna()
                
                # Descriptive statistics
                desc = metric_data.groupby(['Condition', 'Day'])[metric].agg(['mean', 'sem', 'count']).round(3)
                f.write("\nDescriptive Statistics:\n")
                f.write(desc.to_string() + "\n\n")
                
                # Two-Way ANOVA
                try:
                    formula = f"{metric} ~ C(Condition) + C(Day) + C(Condition):C(Day)"
                    model = ols(formula, data=metric_data).fit()
                    anova_table = sm.stats.anova_lm(model, typ=2)
                    
                    f.write("Two-Way ANOVA Results:\n")
                    f.write(anova_table.to_string() + "\n")
                except Exception as e:
                    f.write(f"ANOVA failed: {str(e)}\n")
                    
    print(f"Statistical analysis complete. Results saved to {results_file}")

def visualize_cbx_sdt_metrics(data_dict, results_dir, max_days=5):
    """
    Create line plots for SDT metrics across days.
    """
    metrics = ['d_prime', 'beta', 'cs_plus_correct_rate', 'cs_minus_correct_rate']
    metric_labels = ["d'", 'β', 'CS+ Correct Rate', 'CS- Correct Rate']
    colors = {'CBX': 'seagreen', 'SAL': 'black'}
    
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
            
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'{phase} Phase - SDT Metrics Across Days', fontsize=16, y=0.98)
        axes = axes.ravel()
        
        for metric_idx, (metric, metric_label) in enumerate(zip(metrics, metric_labels)):
            ax = axes[metric_idx]
            
            for condition in ['CBX', 'SAL']:
                if condition not in data_dict[phase]:
                    continue
                    
                daily_values = []
                daily_sems = []
                days_plotted = []
                
                # Extract sorted days up to max_days
                sessions = []
                for session_id in data_dict[phase][condition]:
                    try:
                        day_num = int(session_id.split('-D')[1])
                        if day_num <= max_days:
                            sessions.append((day_num, session_id))
                    except:
                        pass
                sessions.sort()
                
                for day_num, session_id in sessions:
                    day_vals = []
                    for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                        if 'sdt_metrics' in animal_data and metric in animal_data['sdt_metrics']:
                            val = animal_data['sdt_metrics'][metric]
                            if not np.isnan(val) and not np.isinf(val):
                                day_vals.append(val)
                    
                    if day_vals:
                        days_plotted.append(day_num)
                        daily_values.append(np.mean(day_vals))
                        daily_sems.append(stats.sem(day_vals) if len(day_vals) > 1 else 0)
                
                if days_plotted:
                    ax.errorbar(days_plotted, daily_values, yerr=daily_sems, 
                              label=condition, color=colors[condition],
                              marker='o', markersize=8, capsize=5, 
                              linewidth=2, capthick=2)
            
            # Customize subplot
            ax.set_xlabel('Day', fontsize=12)
            ax.set_ylabel(metric_label, fontsize=12)
            ax.set_xticks(range(1, max_days + 1))
            ax.set_xticklabels([f'Day {i}' for i in range(1, max_days + 1)])
            
            # Set specific y-axis limits based on the metric
            if metric == 'd_prime':
                ax.set_ylim(-3, 4)
            elif metric == 'beta':
                ax.set_ylim(-1, 20) # Beta can spike, adjust this limit if your data blows it out
            elif metric in ['cs_plus_correct_rate', 'cs_minus_correct_rate']:
                ax.set_ylim(0, 1.05)
            
            ax.grid(True, linestyle='--', alpha=0.3)
            ax.legend(fontsize=10)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_title(metric_label, fontsize=14, pad=10)
            
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        save_path = results_dir / f'{phase.lower()}_cbx_sdt_metrics_days1to{max_days}.pdf'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

# --- Execution Example ---
if __name__ == "__main__":
    # Assuming 'cbx_data' is your populated dictionary
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'SDT_Analysis'
    results_dir.mkdir(parents=True, exist_ok=True)

    print("Calculating SDT metrics...")
    cbx_data = calculate_cbx_sdt_metrics(cbx_data, threshold=1.5)
    
    print("\nPerforming statistical analyses...")
    analyze_cbx_sdt_metrics(cbx_data, results_dir, max_days=9)

    print("\nCreating SDT metric visualizations...")
    visualize_cbx_sdt_metrics(cbx_data, results_dir, max_days=9)
    
    print("\nSDT Pipeline complete!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def plot_sdt_bar_graphs_select_days(data_dict, results_dir, target_days):
    """
    Creates bar graphs with individual scatter points for SDT metrics on specific days.
    Plots SAL first, then CBX.
    
    target_days format: {'Acquisition': [6, 7], 'Reversal': [1, 2]}
    """
    print(f"\n{'='*60}")
    print("Generating SDT Bar Graphs for Selected Days")
    print(f"{'='*60}")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Extract Target Data
    plot_data = []
    metrics = ['d_prime', 'beta', 'cs_plus_correct_rate', 'cs_minus_correct_rate']
    
    for phase, days in target_days.items():
        if phase not in data_dict:
            continue
            
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
                
            for session_id in data_dict[phase][condition]:
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                    
                if day_num in days:
                    for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                        if 'sdt_metrics' in animal_data:
                            row = {
                                'Phase': phase,
                                'Condition': condition,
                                'Day': day_num,
                                'Animal': animal_id
                            }
                            for metric in metrics:
                                row[metric] = animal_data['sdt_metrics'].get(metric, np.nan)
                            plot_data.append(row)
                            
    if not plot_data:
        print("[!] No data found for the specified target days.")
        return None
        
    df = pd.DataFrame(plot_data)
    
    # 2. Plotting Configuration
    metric_labels = {"d_prime": "d' (Sensitivity)", 
                     "beta": "β (Criterion)", 
                     "cs_plus_correct_rate": "CS+ Correct Rate", 
                     "cs_minus_correct_rate": "CS- Correct Rate"}
                     
    # EXPLICITLY ORDER: SAL first, then CBX
    conditions = ['SAL', 'CBX'] 
    colors = {'SAL': 'gainsboro', 'CBX': 'yellowgreen'}
    
    for phase in df['Phase'].unique():
        phase_df = df[df['Phase'] == phase].copy()
        days_to_plot = sorted(phase_df['Day'].unique())
        
        if not days_to_plot:
            continue
            
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        fig.suptitle(f'{phase} Phase - SDT Metrics (Selected Days)', fontsize=16, y=0.95, fontweight='bold')
        axes = axes.ravel()
        
        for metric_idx, metric in enumerate(metrics):
            ax = axes[metric_idx]
            
            x_ticks_positions = []
            x_ticks_labels = []
            
            # Iterate through the selected days to create grouped bars
            for day_idx, day in enumerate(days_to_plot):
                # Calculate base X position to create gaps between days
                # e.g., Day 6: SAL at 0, CBX at 1. Day 7: SAL at 3, CBX at 4.
                base_x = day_idx * 3 
                x_ticks_positions.append(base_x + 0.5) # Center tick between the two bars
                x_ticks_labels.append(f"Day {int(day)}")
                
                for cond_idx, condition in enumerate(conditions):
                    bar_pos = base_x + cond_idx
                    
                    # Extract values
                    vals = phase_df[(phase_df['Day'] == day) & (phase_df['Condition'] == condition)][metric].dropna().values
                    
                    if len(vals) > 0:
                        mean_val = np.mean(vals)
                        sem_val = stats.sem(vals) if len(vals) > 1 else 0
                        
                        # Plot Bar
                        ax.bar(bar_pos, mean_val, yerr=sem_val, width=0.8,
                               color=colors[condition], edgecolor='black', linewidth=1.5,
                               error_kw=dict(lw=1.5, capsize=5, capthick=1.5, ecolor='black'),
                               zorder=1)
                        
                        # Plot Scatter Points (with jitter)
                        x_jitter = np.random.normal(0, 0.08, size=len(vals))
                        ax.scatter(bar_pos + x_jitter, vals, 
                                   color='black', alpha=0.6, s=40, zorder=2, 
                                   edgecolor='white', linewidth=0.5)
                        
                        # Add 'n' count below the bar for clarity
                        y_text_offset = -0.1 * ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else -0.1
                        ax.text(bar_pos, y_text_offset, f"n={len(vals)}", 
                                ha='center', va='top', fontsize=9, color='gray')

            # Formatting Subplot
            ax.set_title(metric_labels[metric], fontsize=14, pad=10)
            ax.set_xticks(x_ticks_positions)
            ax.set_xticklabels(x_ticks_labels, fontsize=12, fontweight='bold')
            ax.set_ylabel('Metric Value', fontsize=12)
            
            # Metric-specific Y-axis bounds & zero lines
            if metric == 'd_prime':
                ax.set_ylim(-3, 4)
                ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7, zorder=0)
            elif metric == 'beta':
                ax.set_ylim(0, max(10, phase_df['beta'].max() * 1.2)) # Dynamic upper bound for beta spikes
                ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, zorder=0) # Beta=1 is neutral bias
            elif metric in ['cs_plus_correct_rate', 'cs_minus_correct_rate']:
                ax.set_ylim(0, 1.05)
                ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, zorder=0) # Chance line
                
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
            # Custom Legend (only needed on the first subplot to avoid clutter)
            if metric_idx == 0:
                from matplotlib.patches import Patch
                legend_elements = [Patch(facecolor=colors[cond], edgecolor='black', label=cond) for cond in conditions]
                ax.legend(handles=legend_elements, loc='upper right', frameon=False)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        # Save output
        day_str = "_".join(map(str, days_to_plot))
        save_path = results_dir / f'{phase.lower()}_cbx_sdt_bars_days_{day_str}.pdf'
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        print(f"  -> Saved {phase} phase plot to: {save_path.name}")
        
    return df

# --- Execution Example ---
if __name__ == "__main__":
    # Assumes 'cbx_data' is heavily populated from previous steps
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'SDT_Analysis'
    
    # Define exactly which days you want plotted as bar graphs
    targeted_days = {
        'Acquisition': [1, 6, 7, 9],
        'Reversal': [1, 6, 7, 9]
    }
    
    sdt_bar_df = plot_sdt_bar_graphs_select_days(cbx_data, results_dir, target_days=targeted_days)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def export_sdt_metrics_csv(data_dict, results_dir, target_days=None):
    """
    Extracts SDT metrics for the specified target days and exports them to a flat CSV.
    
    target_days format: {'Acquisition': [6, 7], 'Reversal': [1, 2]}
    """
    print(f"\n{'='*50}")
    print("Generating SDT Metrics Summary CSV")
    print(f"{'='*50}")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    metrics = ['d_prime', 'beta', 'cs_plus_correct_rate', 'cs_minus_correct_rate']
    summary_data = []
    
    for phase in ['Acquisition', 'Reversal']:
        if phase not in data_dict:
            continue
            
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
                
            for session_id in data_dict[phase][condition]:
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                    
                # Apply the shared brain filter for target days
                if target_days and phase in target_days:
                    if day_num not in target_days[phase]:
                        continue
                        
                for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                    if 'sdt_metrics' in animal_data:
                        row = {
                            'Phase': phase,
                            'Condition': condition,
                            'Day': day_num,
                            'Animal_ID': animal_id
                        }
                        
                        # Extract each metric safely
                        for metric in metrics:
                            row[metric] = animal_data['sdt_metrics'].get(metric, np.nan)
                            
                        summary_data.append(row)
                        
    # Build DataFrame
    df = pd.DataFrame(summary_data)
    
    if df.empty:
        print("[!] No SDT data available to export for the selected days.")
        return None
        
    # Sort for clean viewing (Phase -> SAL first -> Day -> Animal)
    df = df.sort_values(by=['Phase', 'Condition', 'Day', 'Animal_ID'], 
                        ascending=[True, False, True, True]).reset_index(drop=True)
    
    # Create a smart filename based on the days selected
    if target_days:
        day_strings = []
        for p, d in target_days.items():
            day_strings.append(f"{p[:3]}_{'-'.join(map(str, sorted(d)))}")
        day_suffix = "_days_" + "_".join(day_strings)
    else:
        day_suffix = "_all_days"
        
    output_path = results_dir / f'CBX_SDT_Metrics_Summary{day_suffix}.csv'
    
    # Export
    df.to_csv(output_path, index=False)
    
    print(f"Successfully exported SDT summary to:\n{output_path.name}")
    
    # Display the first few rows to confirm it looks right
    print("\nPreview of exported data:")
    print(df.head())
    
    return df

# --- Execution Example ---
if __name__ == "__main__":
    # Assumes 'cbx_data' is heavily populated from previous steps
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'SDT_Analysis'
    
    # Define exactly which days you want in the CSV (matches your bar graph inputs)
    targeted_days = {
        'Acquisition': [1, 6, 7, 9],
        'Reversal': [1, 6, 7, 9]
    }
    
    sdt_csv_df = export_sdt_metrics_csv(cbx_data, results_dir, target_days=targeted_days)

In [ ]:
##high vs low licking from SDT metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def plot_cbx_licking_conditions_comparison(data_dict, results_dir, max_days=8, threshold=1.5):
    """
    Create stacked bar graphs comparing CBX vs SAL high/low licking responses 
    across consecutive days for Acquisition and Reversal phases.
    """
    print(f"\n===== Creating CBX/SAL Licking Response Visualizations (Days 1-{max_days}) =====")
    
    licking_plots_dir = Path(results_dir) / 'licking_condition_comparisons'
    licking_plots_dir.mkdir(exist_ok=True, parents=True)
    
    # Define colors for visualization
    colors = {
        'CBX_high': 'seagreen',
        'CBX_low': 'palegreen',
        'SAL_high': 'black',
        'SAL_low': 'gainsboro'
    }
    
    # Extract and aggregate counts day by day
    counts = {phase: {} for phase in data_dict.keys()}
    
    for phase in data_dict:
        for condition in ['CBX', 'SAL']:
            if condition not in data_dict[phase]:
                continue
                
            for session_id in data_dict[phase][condition]:
                try:
                    day_num = int(session_id.split('-D')[1])
                except (IndexError, ValueError):
                    continue
                    
                if day_num > max_days:
                    continue
                    
                # Initialize day structure if needed
                if day_num not in counts[phase]:
                    counts[phase][day_num] = {
                        'CBX': {'CS+': {'high': 0, 'low': 0}, 'CS-': {'high': 0, 'low': 0}, 'Combined': {'high': 0, 'low': 0}},
                        'SAL': {'CS+': {'high': 0, 'low': 0}, 'CS-': {'high': 0, 'low': 0}, 'Combined': {'high': 0, 'low': 0}}
                    }
                
                # Count responses for each animal
                for animal_id, animal_data in data_dict[phase][condition][session_id].items():
                    for cue in ['CS+', 'CS-']:
                        if cue in animal_data and 'trial_delta_rates' in animal_data[cue]:
                            rates = animal_data[cue]['trial_delta_rates']
                            high = sum(1 for r in rates if r > threshold)
                            low = len(rates) - high
                            
                            counts[phase][day_num][condition][cue]['high'] += high
                            counts[phase][day_num][condition][cue]['low'] += low
                            
                            counts[phase][day_num][condition]['Combined']['high'] += high
                            counts[phase][day_num][condition]['Combined']['low'] += low

    # Plotting
    for phase in counts:
        if not counts[phase]:
            continue
            
        print(f"\n===== {phase} Phase Analysis =====")
        
        # Sort the days we actually have data for
        valid_days = sorted(list(counts[phase].keys()))
        
        for cue_type in ['CS+', 'CS-', 'Combined']:
            print(f"Plotting {cue_type}...")
            
            plt.figure(figsize=(max(8, len(valid_days) * 2), 6))
            
            # Setting up bar positions
            x_positions = np.arange(len(valid_days)) * 3
            width = 0.8
            
            chi_square_results = {}
            
            for i, day in enumerate(valid_days):
                day_data = counts[phase][day]
                
                # Setup Chi-Square matrix for this day
                contingency_data = []
                
                for j, condition in enumerate(['CBX', 'SAL']):
                    high_count = day_data[condition][cue_type]['high']
                    low_count = day_data[condition][cue_type]['low']
                    total = high_count + low_count
                    
                    if total > 0:
                        contingency_data.append([high_count, low_count])
                        
                        high_pct = (high_count / total) * 100
                        low_pct = (low_count / total) * 100
                        
                        x_pos = x_positions[i] + j * width
                        
                        # Plot Low Licking (Bottom)
                        plt.bar(x_pos, low_pct, width=width, 
                               color=colors[f'{condition}_low'],
                               label=f'{condition} Low' if i == 0 and j == 0 else None)
                        
                        # Plot High Licking (Top)
                        plt.bar(x_pos, high_pct, width=width, 
                               bottom=low_pct,
                               color=colors[f'{condition}_high'],
                               label=f'{condition} High' if i == 0 and j == 0 else None)
                        
                        # Add text inside bars
                        if low_pct > 5:  # Only add text if bar is big enough to fit it
                            plt.text(x_pos, low_pct/2, str(low_count), 
                                   ha='center', va='center', fontsize=9)
                        if high_pct > 5:
                            plt.text(x_pos, low_pct + high_pct/2, str(high_count), 
                                   ha='center', va='center', fontsize=9, color='white')
                        
                        # Add n labels on top
                        plt.text(x_pos, 102, f'n={total}', ha='center', va='bottom', fontsize=8)
                
                # Perform Chi-Square test for the day if both groups have data
                if len(contingency_data) == 2:
                    c_table = np.array(contingency_data)
                    if not np.any(c_table == 0):  # Ensure no zero-counts which break chi2
                        try:
                            chi2, p, dof, expected = stats.chi2_contingency(c_table)
                            chi_square_results[day] = {'chi2': chi2, 'p': p, 'dof': dof}
                            
                            # Add asterisk above the day group if significant
                            if p < 0.05:
                                marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                                plt.text(x_positions[i] + width/2, 110, marker, 
                                         ha='center', va='center', fontsize=14, fontweight='bold')
                        except Exception as e:
                            pass
            
            # Formatting the plot
            plt.xticks([x_positions[i] + width/2 for i in range(len(valid_days))], 
                       [f"Day {d}" for d in valid_days])
            
            plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
            plt.ylabel('Percentage (%)')
            plt.title(f'{phase} Phase: {cue_type} High vs Low Licking')
            plt.ylim(0, 115) 
            
            # Separation lines
            for i in range(1, len(valid_days)):
                plt.axvline(x=(x_positions[i-1] + x_positions[i])/2 + width/2, 
                            color='black', linestyle='--', alpha=0.3)
            
            # Add Chi-square summary text box
            if chi_square_results:
                result_text = "Chi-square Test Results (CBX vs SAL):\n"
                for d, res in chi_square_results.items():
                    sig = " *" if res['p'] < 0.05 else ""
                    result_text += f"Day {d}: χ²({res['dof']})={res['chi2']:.2f}, p={res['p']:.4f}{sig}\n"
                    
                plt.figtext(1.05, 0.05, result_text.strip(), ha='left', va='bottom', fontsize=9,
                           bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
            
            plt.tight_layout()
            
            # Save it
            save_name = f"{phase.lower()}_{cue_type.lower().replace('+', 'plus')}_stacked_licking.pdf"
            plt.savefig(licking_plots_dir / save_name, bbox_inches='tight', dpi=300)
            plt.show()
            
    return licking_plots_dir

# --- Execution Example ---
if __name__ == "__main__":
    # Assuming 'cbx_data' is your populated dictionary
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results'
    
    print("\nCreating CBX vs SAL stacked licking bar charts...")
    plot_cbx_licking_conditions_comparison(cbx_data, results_dir, max_days=9, threshold=1.5)
    print("\nStacked bar charts complete! Check the results directory.")

In [ ]:
###omission test data where sucrose has a 50% probability of following CS+

In [ ]:
import os
import numpy as np
import scipy.io as sio
import scipy.stats as stats
import matplotlib.pyplot as plt
from pathlib import Path

# --- Helper Functions ---

def extract_cues(eventlog):
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    return sucrosecues, nosucrosecues

def calculate_delta_lickrate(cues, licks, baseline_offset=(-3000, -1), period_offset=(1, 3000)):
    if len(cues) == 0:
        return np.array([]), np.nan
        
    baseline_start = cues + baseline_offset[0]
    baseline_end = cues + baseline_offset[1]
    period_start = cues + period_offset[0]
    period_end = cues + period_offset[1]

    baseline_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(baseline_start, baseline_end)])
    period_licking = np.array([np.count_nonzero((licks > start) & (licks < end)) for start, end in zip(period_start, period_end)])

    baseline_hz = baseline_licking / abs(baseline_offset[1] - baseline_offset[0]) * 1000
    period_hz = period_licking / (period_offset[1] - period_offset[0]) * 1000

    delta_lickrate = period_hz - baseline_hz
    return delta_lickrate, np.mean(delta_lickrate)

def calculate_auROC(x, y, offset_to_zero=True):
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan
    U, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    A = U / (len(x) * len(y))
    if offset_to_zero:
        return (2 * (A - 0.5), p)
    else:
        return (A, p)

# --- 1. Omission Dictionary Builder ---

def build_omission_dictionary(basedir, omission_folder_name):
    """
    Extracts data specifically for the Sucrose Omission test day.
    """
    omission_dict = {'CBX': {}, 'SAL': {}}
    target_dir = Path(basedir) / omission_folder_name
    
    if not target_dir.exists():
        print(f"[!] ERROR: Could not find folder '{omission_folder_name}' at {basedir}")
        return omission_dict
        
    print(f"\nProcessing Omission Session from: {omission_folder_name}...")
    
    for matfile in target_dir.glob('*.mat'):
        filename = matfile.name
        clean_name = filename.replace('.mat', '')
        parts = clean_name.split('_')
        
        # Parse Condition and Animal ID
        condition = None
        for c in ['CBX', 'SAL']:
            if c in parts:
                condition = c
                break
                
        if not condition:
            continue
            
        try:
            cond_idx = parts.index(condition)
            animal_id = '_'.join(parts[:cond_idx-1]) 
        except ValueError:
            continue
            
        try:
            behaviordata = sio.loadmat(matfile)
            eventlog = np.squeeze(behaviordata['eventlog'])
            licks = np.squeeze(behaviordata['licks'])
            
            if 'fxdpumps' in behaviordata and behaviordata['fxdpumps'].size > 0:
                fxdpumps = np.squeeze(behaviordata['fxdpumps'])
                if fxdpumps.ndim == 0: 
                    fxdpumps = np.array([fxdpumps])
            else:
                fxdpumps = np.array([])
                
        except Exception as e:
            print(f"  Error loading {filename}: {e}")
            continue
            
        cs_plus_all, cs_minus = extract_cues(eventlog)
        
        # --- SPLIT CS+ TRIALS (Rewarded vs Unrewarded) ---
        cs_plus_rew = []
        cs_plus_unrew = []
        
        for t in cs_plus_all:
            expected_pump_time = t + 3000
            # Ultra-tight 2ms tolerance window to handle Python float quirks
            if fxdpumps.size > 0 and np.any(np.abs(fxdpumps - expected_pump_time) <= 2):
                cs_plus_rew.append(t)
            else:
                cs_plus_unrew.append(t)
                
        cs_plus_rew = np.array(cs_plus_rew)
        cs_plus_unrew = np.array(cs_plus_unrew)
        
        # --- CALCULATE DELTA RATES ---
        # Cue Period (1 to 3000ms) - Animal doesn't know outcome yet
        cs_plus_all_cue_rates, _ = calculate_delta_lickrate(cs_plus_all, licks, period_offset=(1, 3000))
        cs_minus_cue_rates, _ = calculate_delta_lickrate(cs_minus, licks, period_offset=(1, 3000))
        
        # Reward Period (3000 to 6000ms) - Animal experiences outcome
        cs_plus_rew_reward_rates, _ = calculate_delta_lickrate(cs_plus_rew, licks, period_offset=(3000, 6000))
        cs_plus_unrew_reward_rates, _ = calculate_delta_lickrate(cs_plus_unrew, licks, period_offset=(3000, 6000))
        cs_minus_reward_rates, _ = calculate_delta_lickrate(cs_minus, licks, period_offset=(3000, 6000))
        
        # --- auROC (ALL CS+ vs CS-) ---
        auroc_norm, auroc_pval = calculate_auROC(cs_plus_all_cue_rates, cs_minus_cue_rates)
        
        # Store in dictionary
        omission_dict[condition][animal_id] = {
            'auROC_norm': auroc_norm,
            'auROC_pval': auroc_pval,
            'Cue': {
                'CS+_All': cs_plus_all_cue_rates.tolist(),
                'CS-': cs_minus_cue_rates.tolist()
            },
            'Reward': {
                'CS+_Rew': cs_plus_rew_reward_rates.tolist(),
                'CS+_Unrew': cs_plus_unrew_reward_rates.tolist(),
                'CS-': cs_minus_reward_rates.tolist()
            }
        }
        
    return omission_dict

# --- Execution ---
if __name__ == "__main__":
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    omission_folder = 'Sucrose Omission' 
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    omission_data = build_omission_dictionary(basedir, omission_folder)
    
    print("\n--- Overall Discrimination Summary ---")
    for condition in ['CBX', 'SAL']:
        if omission_data[condition]:
            aurocs = [data['auROC_norm'] for data in omission_data[condition].values() if not np.isnan(data['auROC_norm'])]
            if aurocs:
                print(f"{condition} Mean auROC (All CS+ vs CS-): {np.mean(aurocs):.3f} ± {stats.sem(aurocs):.3f}")

In [ ]:
###raster plots for Sucrose Omission
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

# --- Helper Functions ---

def extract_cues(eventlog):
    """Extract CS+ (code 5) and CS- (code 6) timestamps from the event log."""
    sucrosecues = eventlog[eventlog[:, 0] == 5, 1]
    nosucrosecues = eventlog[eventlog[:, 0] == 6, 1]
    return sucrosecues, nosucrosecues

def get_relative_trial_licks(cues, licks, pre_window=3000, post_window=6000):
    """
    Extract lick timestamps relative to cue onset for each trial.
    Returns a list of arrays (one array per trial containing lick times in seconds).
    """
    trial_licks_list = []
    for cue in cues:
        window_licks = licks[(licks > cue - pre_window) & (licks <= cue + post_window)]
        rel_licks_sec = (window_licks - cue) / 1000.0  # align to cue onset, convert to seconds
        trial_licks_list.append(rel_licks_sec)
    return trial_licks_list

# --- 1. Omission Raster Dictionary Builder ---

def build_omission_raster_dictionary(basedir, omission_folder_name):
    """
    Build per-animal raster data for the sucrose-omission session.
    All CS+ trials are kept together in chronological order so the raster
    shows licking dynamics across the session (reward was randomized and
    unsignaled, so the animal can't distinguish trial types in advance).
    """
    raster_dict = {'CBX': {}, 'SAL': {}}
    target_dir = Path(basedir) / omission_folder_name

    if not target_dir.exists():
        print(f"[!] ERROR: Could not find folder '{omission_folder_name}' at {basedir}")
        return raster_dict

    print(f"\nBuilding omission rasters from: {omission_folder_name}...")

    for matfile in target_dir.glob('*.mat'):
        clean_name = matfile.name.replace('.mat', '')
        parts = clean_name.split('_')

        # --- Parse condition + animal ID (same convention as your omission analysis) ---
        condition = None
        for c in ['CBX', 'SAL']:
            if c in parts:
                condition = c
                break
        if not condition:
            continue
        try:
            cond_idx = parts.index(condition)
            animal_id = '_'.join(parts[:cond_idx - 1])
        except ValueError:
            continue

        # --- Load data ---
        try:
            behaviordata = sio.loadmat(matfile)
            eventlog = np.squeeze(behaviordata['eventlog'])
            licks = np.squeeze(behaviordata['licks'])
        except Exception as e:
            print(f"  Error loading {matfile.name}: {e}")
            continue

        cs_plus_all, cs_minus = extract_cues(eventlog)

        # --- Relative lick times per trial (-3s to +6s), chronological order ---
        raster_dict[condition][animal_id] = {
            'Raster_CS_Plus':  get_relative_trial_licks(cs_plus_all, licks),
            'Raster_CS_Minus': get_relative_trial_licks(cs_minus, licks),
        }
        print(f"  {animal_id} ({condition}): {len(cs_plus_all)} CS+, {len(cs_minus)} CS-")

    return raster_dict

# --- 2. Raster Plot Generator ---

def plot_omission_rasters(raster_dict, results_dir, target_animals=None,
                          save_plots=True, show_plots=False):
    """
    One raster figure per animal: CS+ (all trials) | CS-.
    The 3s line marks where reward would arrive on rewarded CS+ trials.

    target_animals : None | str | list[str]
        None -> every animal | 'M1' -> one animal | ['M1','M2'] -> several
    """
    raster_dir = Path(results_dir) / 'Omission_Raster_Plots'
    raster_dir.mkdir(parents=True, exist_ok=True)

    if target_animals is None:
        wanted = None
    elif isinstance(target_animals, str):
        wanted = {target_animals}
    else:
        wanted = set(target_animals)

    # Warn on typos by listing what actually exists
    available_animals = set()
    for condition in ['CBX', 'SAL']:
        available_animals.update(raster_dict.get(condition, {}).keys())
    if wanted is not None:
        missing = wanted - available_animals
        if missing:
            print(f"[!] Warning: requested animals not found: {sorted(missing)}")
            print(f"    Available animals: {sorted(available_animals)}")

    plotted = 0
    for condition in ['SAL', 'CBX']:
        for animal_id, data in raster_dict.get(condition, {}).items():
            if wanted is not None and animal_id not in wanted:
                continue

            cs_plus_trials = data['Raster_CS_Plus']
            cs_minus_trials = data['Raster_CS_Minus']
            if not cs_plus_trials and not cs_minus_trials:
                continue

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
            fig.suptitle(f'{animal_id} | Sucrose Omission | {condition}', fontsize=16)

            def format_raster_ax(ax, trial_data, title, cue_color):
                if trial_data and any(len(t) > 0 for t in trial_data):
                    ax.eventplot(trial_data,
                                 lineoffsets=np.arange(1, len(trial_data) + 1),
                                 linelengths=0.8, color='black', alpha=0.8)
                ax.set_title(f'{title} (n={len(trial_data)})', fontsize=14)
                ax.set_xlabel('Time from Cue Onset (s)', fontsize=12)
                ax.set_xlim(-3, 6)
                ax.axvline(0, color=cue_color, linestyle='-', linewidth=2, alpha=0.6, label='Cue ON')
                ax.axvline(3, color='gray', linestyle='--', linewidth=2, alpha=0.6, label='Reward / Omission')
                num_trials = len(trial_data) if trial_data else 30
                ax.set_ylim(0.5, num_trials + 0.5)
                ax.legend(loc='upper right')

            format_raster_ax(ax1, cs_plus_trials, 'CS+ Trials', 'green')
            ax1.set_ylabel('Trial Number', fontsize=12)
            format_raster_ax(ax2, cs_minus_trials, 'CS- Trials', 'crimson')

            ax1.invert_yaxis()  # trial 1 on top
            plt.tight_layout()

            if save_plots:
                save_name = f"{animal_id}_{condition}_omission_raster.pdf"
                fig.savefig(raster_dir / save_name, bbox_inches='tight', dpi=300)
            if show_plots:
                plt.show()
            else:
                plt.close(fig)
            plotted += 1

    print(f"Done. Generated {plotted} omission raster figure(s). Saved to {raster_dir}")

# --- Execution ---

basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
omission_folder = 'Sucrose Omission'
results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
results_dir.mkdir(parents=True, exist_ok=True)

raster_data = build_omission_raster_dictionary(basedir, omission_folder)

# Safety check
if not raster_data['CBX'] and not raster_data['SAL']:
    print("\n[!] ERROR: No data was extracted. Please check your basedir / omission_folder path!")
else:
    plot_omission_rasters(raster_data, results_dir,
                          target_animals=None,   # None = all | 'M1' | ['M1','M2']
                          save_plots=True, show_plots=False)
    print("\nOmission raster analysis complete!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency

def classify_omission_response(cue_type, is_high_licking):
    """
    Classify response type based on cue type and licking behavior.
    Uses Acquisition rules since Omission tests acquired expectations.
    """
    if cue_type == 'CS+':
        return 'correct response' if is_high_licking else 'incorrect withholding'
    else:  # CS-
        return 'incorrect response' if is_high_licking else 'correct withholding'

def get_significance_stars(p_val):
    """Convert p-value to significance stars."""
    if np.isnan(p_val): return "N/A"
    if p_val < 0.001: return "***"
    elif p_val < 0.01: return "**"
    elif p_val < 0.05: return "*"
    else: return "ns"

def analyze_and_plot_omission_pie_charts(omission_dict, results_dir, threshold=1.5):
    """
    Creates pie charts for response types and runs Chi-Square tests of independence 
    between CBX and SAL within each cue type for the Omission session.
    """
    print(f"\n{'='*60}")
    print("Running Omission SDT Analysis (Pie Charts & Chi-Square)")
    print(f"{'='*60}")
    
    colors_map = {
        'correct response': 'greenyellow',
        'correct withholding': 'greenyellow',
        'incorrect response': 'darkslategrey',
        'incorrect withholding': 'grey'
    }
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    stats_output_file = results_dir / 'CBX_Omission_SDT_ChiSquare_Stats.txt'
    
    with open(stats_output_file, 'w') as f:
        f.write("Omission SDT Analysis: Chi-Square Tests of Independence\n")
        f.write("Comparing CBX vs SAL response distributions within Cue Type.\n")
        f.write("="*70 + "\n\n")
        
    # --- 1. Gather all data for the Omission Day ---
    # We use 'CS+_All' from the Cue period because the animal is anticipating the reward
    day_data = {'SAL': {'CS+': [], 'CS-': []}, 'CBX': {'CS+': [], 'CS-': []}}
    
    for condition in ['SAL', 'CBX']:
        if condition in omission_dict:
            for animal_id, animal_data in omission_dict[condition].items():
                if 'Cue' in animal_data:
                    # Process CS+ (using the consolidated Cue period)
                    if 'CS+_All' in animal_data['Cue']:
                        for rate in animal_data['Cue']['CS+_All']:
                            is_high = rate > threshold
                            resp = classify_omission_response('CS+', is_high)
                            day_data[condition]['CS+'].append(resp)
                            
                    # Process CS-
                    if 'CS-' in animal_data['Cue']:
                        for rate in animal_data['Cue']['CS-']:
                            is_high = rate > threshold
                            resp = classify_omission_response('CS-', is_high)
                            day_data[condition]['CS-'].append(resp)

    # --- 2. Run Chi-Square Stats ---
    p_vals = {'CS+': np.nan, 'CS-': np.nan}
    
    for cue_type in ['CS+', 'CS-']:
        cbx_resps = day_data['CBX'][cue_type]
        sal_resps = day_data['SAL'][cue_type]
        
        if not cbx_resps or not sal_resps:
            continue
            
        unique_resps = sorted(list(set(cbx_resps + sal_resps)))
        
        # Build Contingency Table
        cbx_counts = [cbx_resps.count(r) for r in unique_resps]
        sal_counts = [sal_resps.count(r) for r in unique_resps]
        contingency_table = [cbx_counts, sal_counts]
        
        try:
            chi2, p, dof, expected = chi2_contingency(contingency_table)
            p_vals[cue_type] = p
            stars = get_significance_stars(p)
            
            table_str = f"\n{cue_type} Contingency Table:\n"
            table_str += f"{'':<10} | {unique_resps[0]:<22} | {unique_resps[1] if len(unique_resps)>1 else 'N/A':<22}\n"
            table_str += "-"*60 + "\n"
            table_str += f"{'CBX':<10} | {cbx_counts[0]:<22} | {cbx_counts[1] if len(cbx_counts)>1 else 0:<22}\n"
            table_str += f"{'SAL':<10} | {sal_counts[0]:<22} | {sal_counts[1] if len(sal_counts)>1 else 0:<22}\n"
            table_str += f"Results: Chi2 = {chi2:.3f}, p = {p:.4f} {stars}\n"
            
            print(table_str, end="")
            with open(stats_output_file, 'a') as f: f.write(table_str)
            
        except ValueError:
            err_str = f"\n{cue_type} Contingency Table: Error running stats (likely due to zero variance).\n"
            print(err_str, end="")
            with open(stats_output_file, 'a') as f: f.write(err_str)

    # --- 3. Generate the Pie Charts ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Sucrose Omission - SDT Response Types (SAL vs CBX)', fontsize=16, fontweight='bold', y=1.05)
    
    # Plot SAL on left, CBX on right
    for cond_idx, condition in enumerate(['SAL', 'CBX']):
        ax = axes[cond_idx]
        
        if not day_data[condition]['CS+'] and not day_data[condition]['CS-']:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
            ax.axis('off')
            continue
        
        plot_data = []
        for cue_type in ['CS+', 'CS-']:
            responses = day_data[condition][cue_type]
            if not responses: continue
                
            unique, counts = np.unique(responses, return_counts=True)
            props = dict(zip(unique, counts / len(responses) * 50)) # Scale to 50% half-pie
            
            for resp_type, prop in props.items():
                plot_data.append({
                    'label': f'{cue_type}\n{resp_type}',
                    'size': prop,
                    'color': colors_map[resp_type]
                })
        
        if plot_data:
            sizes = [d['size'] for d in plot_data]
            labels = [d['label'] for d in plot_data]
            colors_list = [d['color'] for d in plot_data]
            
            wedges, texts, autotexts = ax.pie(
                sizes, labels=labels, colors=colors_list,
                autopct='%1.1f%%', startangle=90, labeldistance=1.05
            )
            plt.setp(autotexts, weight='bold')
            plt.setp(texts, size=9)
            
        ax.set_title(f'{condition} Condition', pad=10, fontweight='bold')
        
        # Add significance stars box next to the CBX plot
        if cond_idx == 1: 
            star_text = "Group Comparison:\n"
            star_text += f"CS+: {get_significance_stars(p_vals['CS+'])}\n"
            star_text += f"CS-: {get_significance_stars(p_vals['CS-'])}"
            
            ax.text(1.5, 0, star_text, fontsize=12, fontweight='bold', 
                    ha='left', va='center', bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    plt.tight_layout(rect=[0, 0.03, 0.9, 0.95])
    save_path = results_dir / 'omission_cbx_response_pie_charts.pdf'
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

    print(f"\nOmission SDT text file successfully saved to:\n{stats_output_file}")

# --- Execution Example ---
if __name__ == "__main__":
    # Assumes 'omission_data' is the dictionary you built previously
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    analyze_and_plot_omission_pie_charts(omission_data, results_dir, threshold=1.5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def compute_omission_sdt(cs_plus_rates, cs_minus_rates, threshold=1.5):
    """
    Computes SDT metrics specifically for the Omission data.
    Uses Acquisition rules (CS+ is the target signal).
    """
    cs_plus_total = len(cs_plus_rates)
    cs_minus_total = len(cs_minus_rates)
    
    if cs_plus_total == 0 or cs_minus_total == 0:
        return None
        
    # Calculate Hits and False Alarms
    cs_plus_resp = sum(1 for r in cs_plus_rates if r > threshold)
    cs_minus_resp = sum(1 for r in cs_minus_rates if r > threshold)
    
    hit_rate = cs_plus_resp / cs_plus_total
    fa_rate = cs_minus_resp / cs_minus_total
    
    # Clip to avoid infinite z-scores
    epsilon = 0.001
    hit_rate = np.clip(hit_rate, epsilon, 1 - epsilon)
    fa_rate = np.clip(fa_rate, epsilon, 1 - epsilon)
    
    # Calculate d' and beta
    z_hit = stats.norm.ppf(hit_rate)
    z_fa = stats.norm.ppf(fa_rate)
    
    d_prime = z_hit - z_fa
    beta = np.exp((z_fa**2 - z_hit**2) / 2)
    
    return {
        'd_prime': d_prime,
        'beta': beta,
        'cs_plus_correct_rate': hit_rate,
        'cs_minus_correct_rate': 1 - fa_rate
    }

def plot_omission_sdt_bar_graphs(omission_dict, results_dir, threshold=1.5):
    """
    Calculates SDT metrics and generates individual standalone bar graphs 
    for the Sucrose Omission test. Plots SAL on the left, CBX on the right.
    """
    print(f"\n{'='*60}")
    print("Generating Individual SDT Bar Graphs for Sucrose Omission")
    print(f"{'='*60}")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Extract Data and Calculate SDT on the fly
    plot_data = []
    
    for condition in ['SAL', 'CBX']:
        if condition in omission_dict:
            for animal_id, animal_data in omission_dict[condition].items():
                if 'Cue' in animal_data and 'CS+_All' in animal_data['Cue'] and 'CS-' in animal_data['Cue']:
                    cs_plus_rates = animal_data['Cue']['CS+_All']
                    cs_minus_rates = animal_data['Cue']['CS-']
                    
                    sdt_metrics = compute_omission_sdt(cs_plus_rates, cs_minus_rates, threshold)
                    
                    if sdt_metrics:
                        row = {'Condition': condition, 'Animal': animal_id}
                        row.update(sdt_metrics)
                        plot_data.append(row)
                        
    if not plot_data:
        print("[!] No data found to plot.")
        return None
        
    df = pd.DataFrame(plot_data)
    
    # 2. Plotting Configuration
    metrics = ['d_prime', 'beta', 'cs_plus_correct_rate', 'cs_minus_correct_rate']
    metric_labels = {"d_prime": "d' (Sensitivity)", 
                     "beta": "β (Criterion)", 
                     "cs_plus_correct_rate": "CS+ Correct Rate", 
                     "cs_minus_correct_rate": "CS- Correct Rate"}
                     
    conditions = ['SAL', 'CBX']
    colors = {'SAL': 'gainsboro', 'CBX': 'yellowgreen'}
    
    # Create individual figures for each metric
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(5, 6)) # Taller, narrower ratio for 2 bars
        x_positions = [0, 1]
        
        for i, condition in enumerate(conditions):
            vals = df[df['Condition'] == condition][metric].dropna().values
            
            if len(vals) > 0:
                mean_val = np.mean(vals)
                sem_val = stats.sem(vals) if len(vals) > 1 else 0
                x_pos = x_positions[i]
                
                # Plot Bar
                ax.bar(x_pos, mean_val, yerr=sem_val, width=0.6,
                       color=colors[condition], edgecolor='black', linewidth=1.5,
                       error_kw=dict(lw=1.5, capsize=5, capthick=1.5, ecolor='black'),
                       zorder=1)
                
                # Plot Scatter Points
                x_jitter = np.random.normal(0, 0.05, size=len(vals))
                ax.scatter(x_pos + x_jitter, vals, 
                           color='black', alpha=0.6, s=60, zorder=2, 
                           edgecolor='white', linewidth=0.5)
                
                # Add 'n' count below the bar for clarity
                y_text_offset = -0.1 * ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else -0.1
                ax.text(x_pos, y_text_offset, f"n={len(vals)}", 
                        ha='center', va='top', fontsize=11, color='gray')
                
        # Subplot Formatting
        ax.set_title(f"Sucrose Omission: {metric_labels[metric]}", fontsize=15, pad=15, fontweight='bold')
        ax.set_xticks(x_positions)
        ax.set_xticklabels(conditions, fontsize=13, fontweight='bold')
        ax.set_ylabel(metric_labels[metric], fontsize=13)
        
        # Metric-specific Y-axis bounds & lines
        if metric == 'd_prime':
            ax.set_ylim(-3, 4)
            ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7, zorder=0)
        elif metric == 'beta':
            ax.set_ylim(0, max(5, df['beta'].max() * 1.2)) # Dynamic bound based on actual data
            ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, zorder=0) # Beta=1 is neutral
        elif metric in ['cs_plus_correct_rate', 'cs_minus_correct_rate']:
            ax.set_ylim(0, 1.05)
            ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, zorder=0) # Chance
            
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        
        # Save individual output
        save_path = results_dir / f'omission_cbx_sdt_{metric}_bar.pdf'
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        print(f"  -> Saved {metric} plot to: {save_path.name}")
    
    # Export the calculated metrics to CSV for your records
    csv_path = results_dir / 'CBX_Omission_SDT_Metrics_Summary.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nSDT Data exported to: {csv_path.name}")
    
    return df

# --- Execution Example ---
if __name__ == "__main__":
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    omission_sdt_df = plot_omission_sdt_bar_graphs(omission_data, results_dir, threshold=1.5)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

def compute_omission_sdt(cs_plus_rates, cs_minus_rates, threshold=1.5):
    """
    Computes SDT metrics specifically for the Omission data.
    Uses Acquisition rules (CS+ is the target signal).
    Included here to ensure the cell runs standalone.
    """
    cs_plus_total = len(cs_plus_rates)
    cs_minus_total = len(cs_minus_rates)
    
    if cs_plus_total == 0 or cs_minus_total == 0:
        return None
        
    # Calculate Hits and False Alarms
    cs_plus_resp = sum(1 for r in cs_plus_rates if r > threshold)
    cs_minus_resp = sum(1 for r in cs_minus_rates if r > threshold)
    
    hit_rate = cs_plus_resp / cs_plus_total
    fa_rate = cs_minus_resp / cs_minus_total
    
    # Clip to avoid infinite z-scores
    epsilon = 0.001
    hit_rate = np.clip(hit_rate, epsilon, 1 - epsilon)
    fa_rate = np.clip(fa_rate, epsilon, 1 - epsilon)
    
    # Calculate d' and beta
    z_hit = stats.norm.ppf(hit_rate)
    z_fa = stats.norm.ppf(fa_rate)
    
    d_prime = z_hit - z_fa
    beta = np.exp((z_fa**2 - z_hit**2) / 2)
    
    return {
        'd_prime': d_prime,
        'beta': beta,
        'cs_plus_correct_rate': hit_rate,
        'cs_minus_correct_rate': 1 - fa_rate
    }

def export_omission_sdt_csv(omission_dict, results_dir, threshold=1.5):
    """
    Calculates Omission SDT metrics and exports them directly to a flat CSV.
    """
    print(f"\n{'='*50}")
    print("Generating Omission SDT Metrics Summary CSV")
    print(f"{'='*50}")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    summary_data = []
    
    # Iterate through explicitly to ensure SAL vs CBX processing
    for condition in ['SAL', 'CBX']:
        if condition in omission_dict:
            for animal_id, animal_data in omission_dict[condition].items():
                if 'Cue' in animal_data and 'CS+_All' in animal_data['Cue'] and 'CS-' in animal_data['Cue']:
                    cs_plus_rates = animal_data['Cue']['CS+_All']
                    cs_minus_rates = animal_data['Cue']['CS-']
                    
                    sdt_metrics = compute_omission_sdt(cs_plus_rates, cs_minus_rates, threshold)
                    
                    if sdt_metrics:
                        row = {
                            'Condition': condition,
                            'Animal_ID': animal_id
                        }
                        # Merge the calculated metrics into the row dictionary
                        row.update(sdt_metrics)
                        summary_data.append(row)
                        
    # Build DataFrame
    df = pd.DataFrame(summary_data)
    
    if df.empty:
        print("[!] No Omission SDT data available to export.")
        return None
        
    # Sort for clean viewing (SAL first, then CBX, alphabetically by animal)
    df = df.sort_values(by=['Condition', 'Animal_ID'], 
                        ascending=[False, True]).reset_index(drop=True)
    
    output_path = results_dir / 'CBX_Omission_SDT_Metrics_Summary.csv'
    
    # Export to CSV
    df.to_csv(output_path, index=False)
    
    print(f"Successfully exported Omission SDT summary to:\n{output_path.name}")
    
    # Display the first few rows to confirm it looks right
    print("\nPreview of exported data:")
    print(df.head())
    
    return df

# --- Execution Example ---
if __name__ == "__main__":
    # Assumes 'omission_data' is heavily populated from previous steps
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    omission_sdt_csv_df = export_omission_sdt_csv(omission_data, results_dir, threshold=1.5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def plot_omission_licking_conditions_comparison(omission_dict, results_dir, threshold=1.5):
    """
    Create stacked bar graphs comparing CBX vs SAL high/low licking responses 
    specifically for the Sucrose Omission test day.
    Includes CS+, CS-, and Combined (All Trials) breakdowns.
    """
    print(f"\n{'='*60}")
    print(f"Creating Omission Licking Response Visualizations (Threshold: {threshold} Hz)")
    print(f"{'='*60}")
    
    licking_plots_dir = Path(results_dir) / 'licking_condition_comparisons'
    licking_plots_dir.mkdir(exist_ok=True, parents=True)
    
    # Define colors for visualization
    colors = {
        'CBX_high': 'seagreen',
        'CBX_low': 'palegreen',
        'SAL_high': 'black',
        'SAL_low': 'gainsboro'
    }
    
    # Initialize counts dictionary with the new 'Combined' category
    counts = {
        'CBX': {'CS+_All': {'high': 0, 'low': 0}, 'CS-': {'high': 0, 'low': 0}, 'Combined': {'high': 0, 'low': 0}},
        'SAL': {'CS+_All': {'high': 0, 'low': 0}, 'CS-': {'high': 0, 'low': 0}, 'Combined': {'high': 0, 'low': 0}}
    }
    
    # 1. Extract and aggregate counts
    for condition in ['SAL', 'CBX']:  # Explicitly process SAL then CBX
        if condition in omission_dict:
            for animal_id, animal_data in omission_dict[condition].items():
                if 'Cue' in animal_data:
                    for cue in ['CS+_All', 'CS-']:
                        if cue in animal_data['Cue']:
                            rates = animal_data['Cue'][cue]
                            high = sum(1 for r in rates if r > threshold)
                            low = len(rates) - high
                            
                            # Add to specific cue
                            counts[condition][cue]['high'] += high
                            counts[condition][cue]['low'] += low
                            
                            # Add to combined
                            counts[condition]['Combined']['high'] += high
                            counts[condition]['Combined']['low'] += low

    # 2. Plotting loop now includes 'Combined'
    for cue_type in ['CS+_All', 'CS-', 'Combined']:
        print(f"Plotting {cue_type}...")
        
        plt.figure(figsize=(6, 6))
        
        # Setting up bar positions (SAL at 0, CBX at 1)
        x_positions = [0, 1]
        width = 0.6
        conditions = ['SAL', 'CBX']
        
        contingency_data = []
        
        for i, condition in enumerate(conditions):
            high_count = counts[condition][cue_type]['high']
            low_count = counts[condition][cue_type]['low']
            total = high_count + low_count
            
            if total > 0:
                contingency_data.append([high_count, low_count])
                
                high_pct = (high_count / total) * 100
                low_pct = (low_count / total) * 100
                
                x_pos = x_positions[i]
                
                # Plot Low Licking (Bottom)
                plt.bar(x_pos, low_pct, width=width, 
                        color=colors[f'{condition}_low'], edgecolor='black',
                        label=f'{condition} Low' if i == 0 else None)
                
                # Plot High Licking (Top)
                plt.bar(x_pos, high_pct, width=width, 
                        bottom=low_pct, edgecolor='black',
                        color=colors[f'{condition}_high'],
                        label=f'{condition} High' if i == 0 else None)
                
                # Add text inside bars (raw counts)
                if low_pct > 5:  
                    plt.text(x_pos, low_pct/2, str(low_count), 
                             ha='center', va='center', fontsize=12)
                if high_pct > 5:
                    plt.text(x_pos, low_pct + high_pct/2, str(high_count), 
                             ha='center', va='center', fontsize=12, color='white', fontweight='bold')
                
                # Add n labels on top
                plt.text(x_pos, 102, f'n={total}', ha='center', va='bottom', fontsize=10)
        
        # 3. Perform Chi-Square test
        if len(contingency_data) == 2:
            c_table = np.array(contingency_data)
            if not np.any(c_table == 0):  # Ensure no zero-counts which break chi2
                try:
                    chi2, p, dof, expected = stats.chi2_contingency(c_table)
                    
                    # Add asterisk above the plot if significant
                    if p < 0.05:
                        marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                        plt.text(0.5, 110, marker, ha='center', va='center', fontsize=18, fontweight='bold')
                        
                    # Add Chi-square summary text box
                    sig = " *" if p < 0.05 else ""
                    result_text = f"Chi-square (CBX vs SAL):\nχ²({dof})={chi2:.2f}, p={p:.4f}{sig}"
                    
                    plt.figtext(1.05, 0.5, result_text, ha='left', va='center', fontsize=10,
                               bbox=dict(facecolor='white', alpha=0.8, boxstyle='round'))
                except Exception as e:
                    print(f"  Could not compute Chi-Square for {cue_type}: {e}")

        # Formatting the plot
        plt.xticks(x_positions, conditions, fontsize=12, fontweight='bold')
        plt.ylabel('Percentage of Trials (%)', fontsize=12)
        
        # Determine Title Strings
        if cue_type == 'CS+_All':
            display_cue = "CS+ (All Trials)"
        elif cue_type == 'CS-':
            display_cue = "CS-"
        else:
            display_cue = "Combined (All Trials)"
            
        plt.title(f'Sucrose Omission: {display_cue}\nHigh vs Low Licking', pad=20, fontsize=14)
        
        plt.ylim(0, 115) 
        plt.axhline(y=0, color='black', linewidth=1)
        
        # Legend styling
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=colors['SAL_high'], edgecolor='black', label='SAL High (>1.5Hz)'),
            Patch(facecolor=colors['SAL_low'], edgecolor='black', label='SAL Low'),
            Patch(facecolor=colors['CBX_high'], edgecolor='black', label='CBX High (>1.5Hz)'),
            Patch(facecolor=colors['CBX_low'], edgecolor='black', label='CBX Low')
        ]
        plt.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.05, 0.8))
        
        # Remove top/right borders
        ax = plt.gca()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Save it
        save_name = f"omission_{cue_type.lower().replace('+', 'plus')}_stacked_licking.pdf"
        plt.savefig(licking_plots_dir / save_name, bbox_inches='tight', dpi=300)
        plt.show()
        
    return licking_plots_dir

# --- Execution Example ---
if __name__ == "__main__":
    # Assumes 'omission_data' is heavily populated from previous steps
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    print("\nCreating Omission stacked licking bar charts...")
    plot_omission_licking_conditions_comparison(omission_data, results_dir, threshold=1.5)
    print("\nStacked bar charts complete! Check the results directory.")

In [ ]:
###CSV for omission day with averages per animal for delta lick rates

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def export_omission_summary_csv(omission_dict, results_dir):
    """
    Calculates the average delta lick rate per animal for each period/trial type
    and exports a flat summary CSV including auROC. Updated to match the consolidated
    CS+_All cue period structure.
    """
    print(f"\n{'='*50}")
    print("Generating Omission Summary CSV")
    print(f"{'='*50}")
    
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    summary_data = []
    
    for condition in ['CBX', 'SAL']:
        if condition not in omission_dict:
            continue
            
        for animal_id, data in omission_dict[condition].items():
            
            # Helper function to safely average arrays (handles empty lists gracefully)
            def safe_mean(rates_list):
                return np.mean(rates_list) if len(rates_list) > 0 else np.nan
            
            # Extract and average Cue Period (Consolidated CS+)
            cue_cs_all = safe_mean(data['Cue'].get('CS+_All', []))
            cue_cs_minus = safe_mean(data['Cue'].get('CS-', []))
            
            # Extract and average Reward Period (Split CS+)
            rew_cs_rew = safe_mean(data['Reward'].get('CS+_Rew', []))
            rew_cs_unrew = safe_mean(data['Reward'].get('CS+_Unrew', []))
            rew_cs_minus = safe_mean(data['Reward'].get('CS-', []))
            
            # Append to list
            summary_data.append({
                'Condition': condition,
                'Animal_ID': animal_id,
                'auROC_Norm': data.get('auROC_norm', np.nan),
                'auROC_Pval': data.get('auROC_pval', np.nan),
                'Cue_CS_Plus_All_Avg': cue_cs_all,
                'Cue_CS_Minus_Avg': cue_cs_minus,
                'Reward_CS_Plus_Rewarded_Avg': rew_cs_rew,
                'Reward_CS_Plus_Unrewarded_Avg': rew_cs_unrew,
                'Reward_CS_Minus_Avg': rew_cs_minus
            })
            
    # Build DataFrame
    df = pd.DataFrame(summary_data)
    
    if df.empty:
        print("[!] No data available to export.")
        return None
        
    # Sort for clean viewing (SAL first, then CBX, alphabetically by animal)
    df = df.sort_values(by=['Condition', 'Animal_ID'], ascending=[False, True]).reset_index(drop=True)
    
    # Export
    output_path = results_dir / 'CBX_Omission_Averages_Summary.csv'
    df.to_csv(output_path, index=False)
    
    print(f"Successfully exported average summary to:\n{output_path}")
    
    # Display the first few rows to confirm it looks right
    print("\nPreview of exported data:")
    print(df.head())
    
    return df

# --- Execution ---
if __name__ == "__main__":
    # Assumes 'omission_data' is already in your notebook's memory
    basedir = r'C:\Users\Jackie\Dropbox\Jackie Projects\Projects\2026 PFC CBX Pavlovian Conditioning\Matlab Files and Codes'
    results_dir = Path(basedir) / 'Results' / 'Omission_Analysis'
    
    omission_summary_df = export_omission_summary_csv(omission_data, results_dir)